# Persona 1 — Corpus y metadatos: Glaciares y riesgo glaciar en el Perú (v5 · versión final)
**Proyecto:** Enterprise Brain — Gestión del Conocimiento con Knowledge Graph + GraphRAG · Grupo G8

**Alcance:** documentos sobre glaciares, lagunas de origen glaciar, peligros glaciares, hidrología y ecosistemas de alta montaña **en el Perú**.
**Fuentes:** INAIGEM (principal), ANA y SENAMHI (con filtro temático) · OpenAlex para documentos con DOI.

### Qué produce
| Etapa | Sección | Resultado |
|---|---|---|
| Cosecha de repositorios | 4 | Candidatos con título, resumen, metadatos y enlace al PDF |
| Enriquecimiento | 5 | Autores, ORCID, instituciones y citas (solo documentos con DOI) |
| Acceso a PDF | 5b | Qué documentos se pueden descargar (acceso abierto) |
| Clasificación | 6 | Reglas + similitud semántica → incluir / revisar / excluir |
| Decisiones manuales | 7 | **Todas en una sola celda**, agrupadas por motivo |
| Salidas | 8 | `papers.csv`, `metadata_triples.csv`, `reporte_calidad.md`, etc. |
| PDFs | 9 | Descarga reanudable + `pdfs_manifest.csv` |

### Resultado esperado
**398 documentos** (ANA 184 · INAIGEM 160 · SENAMHI 54), todos con PDF descargable y **0 pendientes de revisión**.

## ▶️ Cómo ejecutarlo
1. Abrir este notebook en **Jupyter en la computadora** (desde Colab la ANA y el SENAMHI no responden).
2. En la sección 0, `CARPETA_LOCAL` debe apuntar a la misma carpeta de siempre (Google Drive para escritorio). Así se reutiliza la caché y los PDFs ya descargados.
3. Menú **Kernel → Restart Kernel and Run All Cells**.
4. Cuando pida `OPENALEX_API_KEY`, pegarla o presionar **Enter** para omitirla.

⏱️ Con la caché existente tarda pocos minutos; solo la sección 9 demora porque descarga los PDFs nuevos de la ANA y el SENAMHI. Si algo se corta, se vuelve a ejecutar y continúa donde quedó.

✏️ Solo hay **dos celdas para editar**: la configuración (sección 2) y las decisiones (sección 7). El resto solo se ejecuta.

🚫 A GitHub se suben el notebook y los CSV/MD de salida. **No** se suben `pdfs/` ni `cache/`.

## 0. Entorno y carpeta de trabajo

In [ ]:
# Carpeta de trabajo en Jupyter (Google Drive para escritorio). Vacío = ~/proyecto_gc/corpus_glaciares
CARPETA_LOCAL = r"G:\Mi unidad\proyecto_gc\corpus_glaciares"
USAR_DRIVE = True   # solo aplica en Colab

import os, sys
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = "/content/drive/MyDrive/proyecto_gc/corpus_glaciares"
elif EN_COLAB:
    CARPETA = "/content/corpus_glaciares"
else:
    CARPETA = (CARPETA_LOCAL or os.environ.get("PROYECTO_GC_CARPETA")
               or os.path.join(os.path.expanduser("~"), "proyecto_gc", "corpus_glaciares"))
os.makedirs(CARPETA, exist_ok=True)

print("Entorno:", "Google Colab" if EN_COLAB else "Jupyter local")
print("Carpeta de trabajo:", CARPETA)
print("Python:", sys.version.split()[0])
previo = [x for x in ("cache", "pdfs", "decisiones_manuales.csv") if os.path.exists(os.path.join(CARPETA, x))]
print("Se reutiliza:", ", ".join(previo) if previo else "nada (carpeta nueva: se descargará todo)")

## 1. Clave de OpenAlex (opcional)
Solo sirve para enriquecer los documentos con DOI. La clave **no se guarda** en el notebook.
- **Jupyter:** se pide al ejecutar (o se lee de la variable de entorno `OPENALEX_API_KEY`).
- **Colab:** ícono 🔑 Secrets → `OPENALEX_API_KEY`.

In [ ]:
import os

def leer_secreto(nombre):
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
        print(f"{nombre}: cargada desde Secrets")
        return valor
    except Exception:
        pass
    if os.environ.get(nombre):
        print(f"{nombre}: cargada desde variable de entorno")
        return os.environ[nombre]
    from getpass import getpass
    valor = getpass(f"Pega tu {nombre} (Enter para omitir): ").strip()
    print(f"{nombre}: " + ("ingresada" if valor else "no configurada (opcional)"))
    return valor or None

OPENALEX_API_KEY = leer_secreto("OPENALEX_API_KEY")

## 2. Configuración ✏️
- `TAMANO_CORPUS`: tamaño final del corpus (398 = resultado acordado tras la revisión).
- `FUENTES_ADICIONALES`: ANA y SENAMHI; para desactivar una, se borra del diccionario.
- `SEMILLAS_HANDLES`: documentos clave que entran siempre.
- `CORRECCIONES`: arreglos manuales de metadatos (por ejemplo, un año mal cargado).

In [ ]:
CONFIG = {
    # Fuente principal: Repositorio INAIGEM (DSpace 7.6.1)
    "REPO_BASE_URL": "https://repositorio.inaigem.gob.pe/server/api",
    "COMUNIDADES": {
        "peligros_glaciar": "7228a39e-e1f6-40d2-9b1b-733231d28adc",
        "dinamica_glaciar": "731bf74b-ff93-465b-a853-41af24bd47e8",
        "gestion_conocimiento": "a6e58cdb-021b-4054-846b-3d79c748cb76",
        "servicios_ecosistemicos": "73f751aa-0f6f-4cf1-b7ff-8b6b66b92293",
        "socioecosistemas": "709c37ea-1350-4c45-8159-6ee377b94dba",
    },
    # Fuentes complementarias (métodos verificados el 28-09-2026)
    "FUENTES_ADICIONALES": {
        "ANA":     {"url": "https://repositorio.ana.gob.pe", "filtro_tematico": True,
                    "metodo": "dspace7", "api": "https://repositorio.ana.gob.pe/backend/api"},
        "SENAMHI": {"url": "https://repositorio.senamhi.gob.pe", "filtro_tematico": True, "metodo": "oai"},
    },
    # Tamaño y selección
    "MAX_CANDIDATOS": 500,
    "TAMANO_CORPUS": 398,
    "MIN_RELEVANCIA": 1,
    "PROPORCION_INVENTARIOS": 0.15,   # ~15 % inventarios / informes técnicos (nodos centrales del grafo)
    "MAX_POR_SERIE": 4,               # máx. documentos de una misma serie (boletines, reportes)
    "SOLO_CON_PDF": True,             # solo documentos con PDF de acceso abierto
    "TIPOS_EXCLUIDOS": ["info:pe-repo/semantics/stillImage", "info:eu-repo/semantics/image"],
    # Clasificador
    "USAR_EMBEDDINGS": True,
    "MODELO_EMBEDDINGS": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "MARGEN_EXCLUIR": 0.04,
    "MARGEN_REVISAR": 0.00,
    # Otros
    "USAR_OPENALEX": True,
    "USAR_CACHE": True,
    "SSL_TOLERANTE": True,            # algunos servidores .gob.pe tienen el certificado incompleto
}

# Documentos clave: entran siempre (solo una decisión manual podría sacarlos)
SEMILLAS_HANDLES = {
    "20.500.12748/57": "Inventario Nacional de Glaciares 2018",
    "20.500.12748/499": "INGLOG 2023 - Inventario Nacional de Glaciares y Lagunas",
}

# Correcciones manuales de metadatos: {handle: {columna: valor}}
CORRECCIONES = {
    "20.500.12748/670": {"year": 2022},   # en el repositorio figura 0027-03-26
}

## 3. Código del pipeline
Solo se ejecuta: instala dependencias y define las funciones. No hace falta modificarla.

In [ ]:
%pip install -q pymupdf sentence-transformers requests pandas scikit-learn truststore

# -*- coding: utf-8 -*-
"""
Persona 1 — Corpus y metadatos: Glaciares y Ecosistemas de Montaña
Proyecto: Enterprise Brain — Gestión del Conocimiento (Knowledge Graph + GraphRAG)
Fuente principal: Repositorio Digital INAIGEM (DSpace 7.6.1)

Etapas:
  1. Descarga de ítems del repositorio INAIGEM via REST API    (con caché)
  2. Enriquecimiento opcional con OpenAlex (docs con DOI)       (con caché)
  3. Clasificación de alcance: reglas + similitud semántica
  4. Decisiones manuales persistentes (se guardan en Drive y se reaplican siempre)
  5. Selección balanceada por año y por eje temático
  6. Salidas: CSV, tripletas, registro de búsqueda, reporte de calidad, diccionario de datos
  7. Descarga de PDFs con manifiesto (reanudable)

Cambios v2:
  - Modelo de embeddings multilingüe y descripciones de ejes en español.
  - Reglas de ejes con raíces (hídric -> hídricos, satélit -> satélites...).
  - Años imposibles (p. ej. 0027) se corrigen con otras fechas del registro; CORRECCIONES manuales.
  - No se infieren afiliaciones: el INAIGEM queda como 'publicadoPor' del documento.
  - Se verifica ANTES de seleccionar si cada PDF es descargable (acceso abierto vs. restringido).
  - Filtro opcional por tipo de documento.
"""
import os, re, json, time, math, hashlib, unicodedata, platform
from datetime import datetime, timezone

import requests
import numpy as np
import pandas as pd

# === URLs base ===
OPENALEX_API = "https://api.openalex.org/works"
HEADERS = {"User-Agent": "UNMSM-GC-Proyecto-Glaciares/1.0 (proyecto academico)"}
VERSION = "5.0-glaciares"


def log(msg):
    print(msg, flush=True)


def ahora():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def limpiar(t):
    return re.sub(r"\s+", " ", t or "").strip()


def titulo_normalizado(t):
    t = unicodedata.normalize("NFKD", t or "").encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", t.lower())


def sin_prefijo(url, prefijo):
    return (url or "").replace(prefijo, "")


# ======================================================================
# HTTP con reintentos
# ======================================================================
class AccesoDenegado(RuntimeError):
    pass


from urllib.parse import urlparse
import urllib3

# Certificados del sistema operativo (Windows / Mac / Linux): evita la mayoría de
# errores SSL con servidores .gob.pe al ejecutar en una computadora personal.
try:
    import truststore
    truststore.inject_into_ssl()
except Exception:
    pass


class SesionTolerante(requests.Session):
    """Sesión HTTP que, si un servidor presenta un certificado SSL incompleto (frecuente en
    dominios .gob.pe), reintenta SIN verificar solo para ese dominio y avisa una vez.
    Se desactiva con CONFIG["SSL_TOLERANTE"] = False."""
    hosts_sin_verificar = set()

    def request(self, method, url, *args, **kwargs):
        host = urlparse(url).hostname or ""
        if host in self.hosts_sin_verificar:
            kwargs["verify"] = False
            return super().request(method, url, *args, **kwargs)
        try:
            return super().request(method, url, *args, **kwargs)
        except requests.exceptions.SSLError:
            if not globals().get("CONFIG", {}).get("SSL_TOLERANTE", True):
                raise
            self.hosts_sin_verificar.add(host)
            urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
            log(f"   ⚠️ {host}: el servidor tiene el certificado SSL incompleto; se continúa sin "
                f"verificarlo (solo este dominio; son documentos de acceso público).")
            kwargs["verify"] = False
            return super().request(method, url, *args, **kwargs)


_SESION = SesionTolerante()


def http(metodo, url, params=None, json_body=None, headers=None, intentos=5, espera=4, timeout=90):
    """GET/POST con reintentos. 429 y 5xx -> espera creciente. 401/403 -> error claro."""
    h = dict(HEADERS, **(headers or {}))
    for i in range(intentos):
        try:
            r = _SESION.request(metodo, url, params=params, json=json_body, headers=h, timeout=timeout)
        except requests.RequestException as e:
            log(f"   Error de red ({e.__class__.__name__}); reintento {i + 1}/{intentos}")
            time.sleep(espera * (i + 1))
            continue
        if r.status_code == 200:
            return r
        if r.status_code in (401, 403):
            raise AccesoDenegado(f"{url.split('?')[0]} respondió {r.status_code}: {r.text[:200]}")
        if r.status_code in (400, 404):
            return r
        pausa = espera * (2 ** i) if r.status_code == 429 else espera * (i + 1)
        log(f"   HTTP {r.status_code} en {url.split('?')[0]}; espero {pausa}s ({i + 1}/{intentos})")
        time.sleep(pausa)
    raise RuntimeError(f"No se pudo consultar {url.split('?')[0]} tras {intentos} intentos")


# ======================================================================
# Caché en disco (JSONL por fuente; ideal para Drive: pocos archivos)
# ======================================================================
class Cache:
    """Guarda cada respuesta de las APIs. Volver a ejecutar el notebook no repite
    consultas, es mucho más rápido y el corpus es reproducible."""

    def __init__(self, carpeta, activa=True):
        self.dir = carpeta
        self.activa = activa
        self._mem = {}
        os.makedirs(carpeta, exist_ok=True)

    def _ruta(self, espacio):
        return os.path.join(self.dir, f"{espacio}.jsonl")

    def _cargar(self, espacio):
        if espacio in self._mem:
            return self._mem[espacio]
        d = {}
        ruta = self._ruta(espacio)
        if os.path.exists(ruta):
            with open(ruta, encoding="utf-8") as f:
                for linea in f:
                    try:
                        o = json.loads(linea)
                        d[o["k"]] = o["v"]
                    except Exception:
                        pass
        self._mem[espacio] = d
        return d

    def tiene(self, espacio, clave):
        return self.activa and clave in self._cargar(espacio)

    def get(self, espacio, clave, defecto=None):
        return self._cargar(espacio).get(clave, defecto) if self.activa else defecto

    def put_muchos(self, espacio, pares):
        d = self._cargar(espacio)
        with open(self._ruta(espacio), "a", encoding="utf-8") as f:
            for k, v in pares:
                d[k] = v
                f.write(json.dumps({"k": k, "v": v}, ensure_ascii=False) + "\n")

    def put(self, espacio, clave, valor):
        self.put_muchos(espacio, [(clave, valor)])

    def resumen(self):
        return {e.replace(".jsonl", ""): sum(1 for _ in open(os.path.join(self.dir, e), encoding="utf-8"))
                for e in sorted(os.listdir(self.dir)) if e.endswith(".jsonl")}


# ======================================================================
# 1. Repositorio INAIGEM (DSpace 7.6.1)
# ======================================================================
def parsear_item_dspace(item, meta):
    """Extrae metadatos de un ítem DSpace en formato compatible con el pipeline."""
    def meta_val(campo, idx=0):
        vals = meta.get(campo, [])
        return vals[idx]["value"] if vals and idx < len(vals) else ""

    def meta_vals(campo):
        return [v["value"] for v in meta.get(campo, [])]

    return {
        "item_id": item.get("uuid", ""),
        "handle": item.get("handle", ""),
        "title": limpiar(meta_val("dc.title")),
        "abstract": limpiar(meta_val("dc.description.abstract")),
        "published_date": meta_val("dc.date.issued"),
        "date_available": meta_val("dc.date.available"),
        "date_accessioned": meta_val("dc.date.accessioned"),
        "citation": meta_val("dc.identifier.citation"),
        "authors_repo": meta_vals("dc.contributor.author"),
        "subjects": meta_vals("dc.subject"),
        "doc_type": limpiar(meta_val("dc.type")),
        "language": meta_val("dc.language.iso"),
        "doi": meta_val("dc.identifier.doi"),
        "publisher": meta_val("dc.publisher"),
        "rights": meta_val("dc.rights"),
        "uri": meta_val("dc.identifier.uri"),
        "community": "",  # se llena al recorrer comunidades
        "fuente": "INAIGEM",
    }


def buscar_repositorio_inaigem(base_url, cache, max_items=500, por_pagina=100, pausa=1.0):
    """Descarga todos los ítems del repositorio INAIGEM via REST API de DSpace 7."""
    registros = []
    pagina = 0
    total = None
    while total is None or pagina * por_pagina < min(total, max_items):
        # (se mantiene la clave antigua para el INAIGEM y así reutilizar su caché)
        clave = (f"inaigem_items|{pagina}|{por_pagina}" if "inaigem" in base_url
                 else f"items|{base_url}|{pagina}|{por_pagina}")
        if cache.tiene("inaigem", clave):
            datos = cache.get("inaigem", clave)
        else:
            url = f"{base_url}/discover/search/objects"
            params = {
                "query": "*",
                "dsoType": "ITEM",
                "size": por_pagina,
                "page": pagina,
                "sort": "dc.date.accessioned,DESC",
            }
            r = http("GET", url, params=params)
            if r.status_code != 200:
                log(f"   ⚠️ Error HTTP {r.status_code} en página {pagina}")
                break
            datos = r.json()
            cache.put("inaigem", clave, datos)
            time.sleep(pausa)

        if total is None:
            total = (datos.get("_embedded", {})
                     .get("searchResult", {})
                     .get("page", {})
                     .get("totalElements", 0))
            log(f"Repositorio {'INAIGEM' if 'inaigem' in base_url else base_url.split('/')[2]}: {total} ítems en total")

        items = (datos.get("_embedded", {})
                 .get("searchResult", {})
                 .get("_embedded", {})
                 .get("objects", []))
        for obj in items:
            item = obj.get("_embedded", {}).get("indexableObject", {})
            meta = item.get("metadata", {})
            registros.append(parsear_item_dspace(item, meta))

        pagina += 1
        log(f"   {len(registros)} / {min(total, max_items)} descargados")

    return registros, total or 0


def asignar_comunidades(registros, base_url, comunidades, cache, pausa=1.0):
    """Asigna la comunidad (línea de investigación) a cada ítem consultando
    las colecciones de cada comunidad."""
    coleccion_a_comunidad = {}
    for nombre, uuid in comunidades.items():
        clave = f"comunidad_cols|{uuid}"
        if cache.tiene("inaigem", clave):
            cols = cache.get("inaigem", clave)
        else:
            url = f"{base_url}/core/communities/{uuid}/collections"
            r = http("GET", url)
            if r.status_code != 200:
                log(f"   ⚠️ No se pudieron obtener colecciones de {nombre}")
                continue
            cols = r.json()
            cache.put("inaigem", clave, cols)
            time.sleep(pausa)
        for col in cols.get("_embedded", {}).get("collections", []):
            coleccion_a_comunidad[col["uuid"]] = nombre

    # Ahora asignar a cada registro su comunidad
    handle_a_reg = {r["handle"]: r for r in registros}
    for nombre, uuid in comunidades.items():
        # Buscar ítems por comunidad
        clave = f"comunidad_items|{uuid}"
        if cache.tiene("inaigem", clave):
            items_com = cache.get("inaigem", clave)
        else:
            url = f"{base_url}/discover/search/objects"
            params = {
                "query": "*",
                "dsoType": "ITEM",
                "scope": uuid,
                "size": 200,
                "page": 0,
            }
            r = http("GET", url, params=params)
            if r.status_code == 200:
                items_com = r.json()
                cache.put("inaigem", clave, items_com)
            else:
                items_com = {}
            time.sleep(pausa)

        objs = (items_com.get("_embedded", {})
                .get("searchResult", {})
                .get("_embedded", {})
                .get("objects", []))
        for obj in objs:
            item = obj.get("_embedded", {}).get("indexableObject", {})
            h = item.get("handle", "")
            if h in handle_a_reg and not handle_a_reg[h]["community"]:
                handle_a_reg[h]["community"] = nombre

    asignados = sum(1 for r in registros if r["community"])
    log(f"Comunidades asignadas: {asignados}/{len(registros)}")
    return registros


def buscar_semillas_inaigem(handles, registros):
    """Marca como semilla los documentos cuyos handles están en la lista."""
    handle_set = set(handles.keys())
    encontradas = []
    for r in registros:
        if r["handle"] in handle_set:
            r["semilla"] = True
            encontradas.append(r["handle"])
        else:
            r.setdefault("semilla", False)
    faltan = handle_set - set(encontradas)
    if faltan:
        log(f"   ⚠️ Semillas no encontradas (revisar handle): {faltan}")
    log(f"Semillas encontradas: {len(encontradas)}/{len(handles)}")
    return registros


def deduplicar(registros):
    vistos_id, vistos_t, salida = set(), set(), []
    for r in registros:
        tn = titulo_normalizado(r["title"])
        uid = r.get("item_id") or r.get("handle") or tn
        if uid in vistos_id or tn in vistos_t:
            continue
        vistos_id.add(uid)
        vistos_t.add(tn)
        salida.append(r)
    return salida


# ======================================================================
# 1b. Fuentes adicionales (ANA, SENAMHI...): detección automática del método
#     DSpace 7 (REST /server/api)  ->  OAI-PMH (/oai/request)  ->  DSpace 6 (REST /rest)
# ======================================================================
import xml.etree.ElementTree as ET
from html import unescape

NS_OAI = {"oai": "http://www.openarchives.org/OAI/2.0/",
          "oai_dc": "http://www.openarchives.org/OAI/2.0/oai_dc/",
          "dc": "http://purl.org/dc/elements/1.1/"}

# Filtro temático para fuentes que NO son solo de glaciares (la ANA y el SENAMHI publican
# sobre agua y clima en general): el documento debe mencionar glaciares o alta montaña.
PAT_FILTRO_EXTERNO = re.compile(
    r"\b(glaciar\w*|glacier\w*|glaciol\w*|nevado\w*|criosf\w*|cryospher\w*|deshielo\w*|"
    r"lagunas? (de origen )?glaciar\w*|aluvi[oó]n\w*|GLOF\w*|bofedal\w*|altoandin\w*|"
    r"alta monta[nñ]a|ecosistemas? de monta[nñ]a|cordillera (blanca|negra|huayhuash|vilcanota|"
    r"vilcabamba|central|raura|huallanca|ampato|chila|la raya|huaytapallana|urubamba|carabaya))\b", re.I)


def _get_texto(url, params=None, intentos=3):
    try:
        r = http("GET", url, params=params, intentos=intentos, espera=3, timeout=60)
        return r.status_code, r.text if hasattr(r, "text") else ""
    except AccesoDenegado:
        return 401, ""
    except Exception:
        return 0, ""


def detectar_metodo(url, cache):
    """Prueba qué API responde en un repositorio DSpace y recuerda el resultado."""
    clave = f"metodo2|{url}"
    if cache.tiene("fuentes", clave):
        return tuple(cache.get("fuentes", clave))
    metodo, api = None, ""
    for ruta in ("/server/api", "/backend/api"):   # la ANA usa /backend/api
        st, txt = _get_texto(f"{url}{ruta}/discover/search/objects", {"query": "*", "dsoType": "ITEM", "size": 1})
        if st == 200 and "searchResult" in txt:
            metodo, api = "dspace7", f"{url}{ruta}"
            break
    if metodo is None:
        st, txt = _get_texto(f"{url}/oai/request", {"verb": "Identify"})
        if st == 200 and "<Identify" in txt:
            metodo = "oai"
    if metodo is None:
        st, txt = _get_texto(f"{url}/rest/items", {"limit": 1})
        if st == 200 and txt.strip().startswith("["):
            metodo = "rest6"
    if metodo:
        cache.put("fuentes", clave, [metodo, api])
    return metodo, api


def _registro_desde_campos(campos, handle, item_id, fuente):
    """Convierte {'dc.title': [...], ...} al formato de parsear_item_dspace."""
    meta = {k: [{"value": v} for v in vs] for k, vs in campos.items()}
    r = parsear_item_dspace({"uuid": item_id, "handle": handle}, meta)
    r["fuente"] = fuente
    return r


def cosechar_oai(url, fuente, cache, max_items=20000, pausa=1.0):
    """Cosecha todos los registros en Dublin Core por OAI-PMH (páginas con resumptionToken)."""
    registros, token, pagina = [], None, 0
    while len(registros) < max_items:
        params = {"verb": "ListRecords", "resumptionToken": token} if token else \
                 {"verb": "ListRecords", "metadataPrefix": "oai_dc"}
        clave = f"oai|{url}|{token or 'inicio'}"
        if cache.tiene("fuentes", clave):
            xml_txt = cache.get("fuentes", clave)
        else:
            st, xml_txt = _get_texto(f"{url}/oai/request", params)
            if st != 200:
                log(f"   ⚠️ {fuente}: OAI respondió {st} en la página {pagina}")
                break
            cache.put("fuentes", clave, xml_txt)
            time.sleep(pausa)
        raiz = ET.fromstring(xml_txt.encode("utf-8"))
        err = raiz.find("oai:error", NS_OAI)
        if err is not None:
            if err.get("code") != "noRecordsMatch":
                log(f"   ⚠️ {fuente}: OAI error {err.get('code')}: {err.text}")
            break
        for rec in raiz.iterfind(".//oai:record", NS_OAI):
            head = rec.find("oai:header", NS_OAI)
            if head is None or head.get("status") == "deleted":
                continue
            ident = head.findtext("oai:identifier", default="", namespaces=NS_OAI)
            handle = ident.split(":")[-1] if re.search(r"\d+\.\d+/\d+", ident.split(":")[-1]) else ""
            dc = rec.find(".//oai_dc:dc", NS_OAI)
            if dc is None:
                continue
            val = lambda tag: [limpiar(e.text) for e in dc.findall(f"dc:{tag}", NS_OAI) if e.text and e.text.strip()]
            fechas = val("date")
            anios = sorted(a for a in (extraer_anio(f) for f in fechas) if es_anio_valido(a))
            descr = sorted(val("description"), key=len, reverse=True)
            idents = val("identifier")
            doi = next((m.group(0).rstrip(".").lower() for i in idents for m in [re.search(r"10\.\d{4,9}/\S+", i)] if m), "")
            uri = next((i for i in idents if i.startswith("http")), "")
            if not handle:
                m = re.search(r"(\d+\.\d+/\d+)\s*$", uri)
                handle = m.group(1) if m else ""
            campos = {"dc.title": val("title")[:1], "dc.description.abstract": descr[:1],
                      "dc.date.issued": [str(anios[0])] if anios else fechas[:1],
                      "dc.contributor.author": val("creator"), "dc.subject": val("subject"),
                      "dc.type": val("type")[:1], "dc.language.iso": val("language")[:1],
                      "dc.identifier.doi": [doi] if doi else [], "dc.publisher": val("publisher")[:1],
                      "dc.rights": val("rights")[:1], "dc.identifier.uri": [uri] if uri else []}
            registros.append(_registro_desde_campos(campos, handle, ident, fuente))
        token = (raiz.findtext(".//oai:resumptionToken", default="", namespaces=NS_OAI) or "").strip()
        pagina += 1
        if pagina % 10 == 0:
            log(f"   {fuente}: {len(registros)} registros cosechados")
        if not token:
            break
    return registros


def cosechar_rest6(url, fuente, cache, max_items=20000, por_pagina=100, pausa=1.0):
    """DSpace 5/6: /rest/items con metadatos y archivos."""
    registros, offset = [], 0
    while offset < max_items:
        clave = f"rest6|{url}|{offset}"
        if cache.tiene("fuentes", clave):
            items = cache.get("fuentes", clave)
        else:
            try:
                r = http("GET", f"{url}/rest/items", params={"expand": "metadata,bitstreams",
                                                           "limit": por_pagina, "offset": offset}, intentos=3)
                items = r.json() if r.status_code == 200 else []
            except Exception as e:
                log(f"   ⚠️ {fuente}: REST falló en offset {offset}: {str(e)[:100]}")
                break
            cache.put("fuentes", clave, items)
            time.sleep(pausa)
        if not items:
            break
        for it in items:
            campos = {}
            for m in it.get("metadata", []):
                campos.setdefault(m.get("key", ""), []).append(m.get("value", ""))
            reg = _registro_desde_campos(campos, it.get("handle", ""), it.get("uuid") or str(it.get("id", "")), fuente)
            pdfs = [b for b in (it.get("bitstreams") or [])
                    if (b.get("bundleName") == "ORIGINAL") and b.get("name", "").lower().endswith(".pdf")]
            if pdfs:
                b = max(pdfs, key=lambda x: x.get("sizeBytes") or 0)
                reg["pdf_url_hint"] = url + b.get("retrieveLink", "")
            registros.append(reg)
        offset += len(items)
        if (offset // por_pagina) % 10 == 0:
            log(f"   {fuente}: {len(registros)} registros cosechados")
    return registros


def fuente_alcanzable(url, timeout=20):
    """Prueba rápida (una sola vez) para no esperar minutos si el servidor no responde."""
    try:
        _SESION.get(url, headers=HEADERS, timeout=timeout, stream=True).close()
        return True, ""
    except requests.exceptions.ConnectTimeout:
        return False, "el servidor no respondió (ConnectTimeout)"
    except requests.RequestException as e:
        return False, e.__class__.__name__
    except Exception:
        return True, ""  # error inesperado en la prueba: se intenta cosechar igual


def cosechar_fuente(nombre, cfg, cache):
    """Cosecha una fuente adicional. Si falla, avisa y devuelve una lista vacía:
    las fuentes complementarias nunca detienen el notebook."""
    try:
        return _cosechar_fuente(nombre, cfg, cache)
    except Exception as e:
        log(f"   ⚠️ {nombre}: la cosecha se interrumpió ({str(e)[:150]}). Se continúa sin esta fuente.")
        return [], {"metodo": cfg.get("metodo"), "cosechados": 0, "del_tema": 0, "error": str(e)[:150]}


def _cosechar_fuente(nombre, cfg, cache):
    url = cfg["url"].rstrip("/")
    if cfg.get("activa", True) is False:
        log(f"{nombre}: desactivada en la configuración.")
        return [], {"metodo": None, "cosechados": 0, "del_tema": 0, "error": "desactivada"}
    # si ya se cosechó antes (caché), no hace falta conectarse de nuevo
    ya_en_cache = cache.activa and any(url in k for esp in ("fuentes", "inaigem") for k in cache._cargar(esp))
    if not ya_en_cache:
        ok_red, motivo = fuente_alcanzable(url)
        if not ok_red:
            donde = ("Desde Colab no se puede acceder a este repositorio (probablemente solo acepta "
                     "conexiones desde el Perú): ejecuten el notebook en Jupyter desde su computadora"
                     if globals().get("EN_COLAB") else
                     "Tampoco responde desde esta computadora: comprueben que la página abra en el navegador, "
                     "desactiven VPN o proxy, o prueben otra red (por ejemplo, datos móviles)")
            log(f"   ⚠️ {nombre}: {motivo}. {donde}. Se continúa sin esta fuente.")
            return [], {"metodo": cfg.get("metodo"), "cosechados": 0, "del_tema": 0, "error": motivo}
    if cfg.get("metodo"):
        metodo = cfg["metodo"]
        api = cfg.get("api", f"{url}/server/api") if metodo == "dspace7" else ""
    else:
        metodo, api = detectar_metodo(url, cache)
    log(f"{nombre}: método detectado = {metodo or 'ninguno'} ({api or url})")
    if metodo == "dspace7":
        regs, _ = buscar_repositorio_inaigem(api, cache, max_items=cfg.get("max_items", 20000))
        for r in regs:
            r["fuente"] = nombre
    elif metodo == "oai":
        regs = cosechar_oai(url, nombre, cache, cfg.get("max_items", 20000))
    elif metodo == "rest6":
        regs = cosechar_rest6(url, nombre, cache, cfg.get("max_items", 20000))
    else:
        log(f"   ⚠️ {nombre}: no respondió ninguna API (DSpace 7, OAI-PMH ni REST 6). Se omite esta fuente.")
        return [], {"metodo": None, "cosechados": 0, "del_tema": 0}
    for r in regs:
        r["fuente_url"], r["fuente_metodo"], r["fuente_api"] = url, metodo, api
    total = len(regs)
    if cfg.get("filtro_tematico", True):
        regs = [r for r in regs if PAT_FILTRO_EXTERNO.search(
            f"{r['title']} {r.get('abstract', '')} {' '.join(r.get('subjects', []))}")]
    log(f"   {nombre}: {total} registros cosechados → {len(regs)} sobre glaciares / alta montaña")
    return regs, {"metodo": metodo, "cosechados": total, "del_tema": len(regs)}


def pdf_desde_rest_handle(url_base, handle, cache, pausa=0.3):
    """DSpace 5/6: /rest/handle/<handle>?expand=bitstreams devuelve los archivos del documento."""
    clave = f"rest_handle|{url_base}|{handle}"
    if cache.tiene("fuentes", clave):
        return cache.get("fuentes", clave)
    try:
        r = http("GET", f"{url_base}/rest/handle/{handle}", params={"expand": "bitstreams"}, intentos=2, espera=3)
    except Exception:
        return None
    if r.status_code != 200:
        return None
    try:
        bits = r.json().get("bitstreams") or []
    except Exception:
        return None
    pdfs = [b for b in bits if b.get("bundleName") == "ORIGINAL" and (b.get("name") or "").lower().endswith(".pdf")]
    info = {"url": "", "nombre": "", "bytes": None}
    if pdfs:
        b = max(pdfs, key=lambda x: x.get("sizeBytes") or 0)
        info = {"url": url_base + b.get("retrieveLink", ""), "nombre": b.get("name", ""), "bytes": b.get("sizeBytes")}
    cache.put("fuentes", clave, info)
    time.sleep(pausa)
    return info


def pdf_desde_pagina(url_base, handle, cache, pausa=0.5):
    """Busca el PDF en la página del documento (meta citation_pdf_url o enlace /bitstream/...pdf)."""
    clave = f"pagina_pdf|{url_base}|{handle}"
    if cache.tiene("fuentes", clave):
        return cache.get("fuentes", clave)
    st, html = _get_texto(f"{url_base}/handle/{handle}")
    if st != 200:
        return None  # error: se reintentará
    m = (re.search(r'<meta[^>]+name="citation_pdf_url"[^>]+content="([^"]+)"', html, re.I) or
         re.search(r'<meta[^>]+content="([^"]+)"[^>]+name="citation_pdf_url"', html, re.I) or
         re.search(r'href="([^"]*/bitstream/[^"]+?\.pdf[^"]*)"', html, re.I))
    enlace = unescape(m.group(1)) if m else ""
    if enlace.startswith("/"):
        enlace = url_base + enlace
    info = {"url": enlace, "nombre": enlace.split("/")[-1].split("?")[0], "bytes": None}
    cache.put("fuentes", clave, info)
    time.sleep(pausa)
    return info


def clave_serie(titulo):
    """Agrupa documentos de una misma serie (boletines numerados, reportes mensuales...)."""
    t = unicodedata.normalize("NFKD", (titulo or "").lower()).encode("ascii", "ignore").decode()
    t = re.sub(r"\b(enero|febrero|marzo|abril|mayo|junio|julio|agosto|se[pt]tiembre|octubre|noviembre|diciembre|"
               r"n|no|nro|numero|vol|volumen|edicion|semestre|trimestre|mes)\b", " ", t)
    t = re.sub(r"[\d\W_]+", "", t)
    return t[:80]


# ======================================================================
# 2. OpenAlex (enriquecimiento para docs con DOI)
# ======================================================================
CAMPOS_OA = ("id,doi,title,publication_year,cited_by_count,authorships,primary_location,"
             "locations,referenced_works,primary_topic,open_access")
TIPOS_VENUE = ("journal", "conference", "book series", "ebook platform")


def consultar_openalex(dois, api_key, cache, lote=50, pausa=0.15):
    dois = sorted(set(d.lower() for d in dois if d))
    if not dois:
        log("OpenAlex: ningún documento tiene DOI; se omite el enriquecimiento.")
        return {}
    faltan = [d for d in dois if not cache.tiene("openalex", d)]
    log(f"OpenAlex: {len(dois) - len(faltan)} DOIs en caché, {len(faltan)} por consultar")
    for i in range(0, len(faltan), lote):
        bloque = faltan[i:i + lote]
        params = {"filter": "doi:" + "|".join(bloque), "per_page": lote, "select": CAMPOS_OA}
        if api_key:
            params["api_key"] = api_key
        r = http("GET", OPENALEX_API, params=params)
        encontrados = {}
        for w in r.json().get("results", []):
            d = sin_prefijo((w.get("doi") or "").lower(), "https://doi.org/")
            if d:
                encontrados[d] = w
        cache.put_muchos("openalex", [(d, encontrados.get(d)) for d in bloque])
        if (i // lote) % 5 == 0 or i + lote >= len(faltan):
            log(f"   OpenAlex: {min(i + lote, len(faltan))} / {len(faltan)} consultados")
        time.sleep(pausa)
    return {d: cache.get("openalex", d) for d in dois}


def venue_openalex(w):
    for loc in [(w or {}).get("primary_location")] + ((w or {}).get("locations") or []):
        src = (loc or {}).get("source") or {}
        if src.get("display_name") and src.get("type") in TIPOS_VENUE:
            return src["display_name"]
    return ""


# ======================================================================
# Integración de metadatos
# ======================================================================
ANIO_MIN, ANIO_MAX = 1950, datetime.now().year + 1


def extraer_anio(texto):
    m = re.search(r"(?<!\d)(\d{4})(?!\d)", str(texto or ""))
    return int(m.group(1)) if m else None


def es_anio_valido(a):
    return a is not None and ANIO_MIN <= a <= ANIO_MAX


def primer_anio(texto):
    m = re.search(r"(?<!\d)(19[5-9]\d|20\d{2})(?!\d)", str(texto or ""))
    return int(m.group(1)) if m else None


def anio_valido(r, w=None):
    """Devuelve (año, fuente) con el primer valor plausible en este orden:
    OpenAlex, dc.date.issued, cita bibliográfica, título, dc.date.available, dc.date.accessioned."""
    candidatos = [
        ((w or {}).get("publication_year"), "openalex"),
        (extraer_anio(r.get("published_date")), "dc.date.issued"),
        (primer_anio(r.get("citation")), "dc.identifier.citation"),
        (primer_anio(r.get("title")), "titulo"),
        (extraer_anio(r.get("date_available")), "dc.date.available"),
        (extraer_anio(r.get("date_accessioned")), "dc.date.accessioned"),
    ]
    for a, fuente in candidatos:
        if es_anio_valido(a):
            return a, fuente
    return None, ""


def integrar_metadatos(registros, works_oa):
    """Integra metadatos de INAIGEM con los de OpenAlex (cuando hay DOI)."""
    autorias = []
    for r in registros:
        doi = (r.get("doi") or "").lower().strip()
        w = works_oa.get(doi) if doi else None

        r["paper_id"] = r["handle"] or r["item_id"]
        r["openalex_id"] = sin_prefijo((w or {}).get("id"), "https://openalex.org/")
        r["en_openalex"] = w is not None

        # Año (se descartan fechas imposibles como 0027 y se busca en otros campos)
        r["year_repo"] = extraer_anio(r.get("published_date"))
        r["year"], r["year_fuente"] = anio_valido(r, w)

        # Citas
        r["cited_by_count"] = (w or {}).get("cited_by_count") or 0
        r["cited_by_fuente"] = "openalex" if r["cited_by_count"] else ""

        # Venue
        v_oa = venue_openalex(w) if w else ""
        r["venue"] = v_oa or r.get("publisher") or f"Repositorio {r.get('fuente', 'INAIGEM')}"
        r["venue_fuente"] = "openalex" if v_oa else ("inaigem" if r.get("publisher") else "repositorio")

        # Referencias
        refs = set()
        if w:
            refs.update(sin_prefijo(u, "https://openalex.org/") for u in (w or {}).get("referenced_works", []))
        r["referenced_works"] = sorted(refs)

        r["openalex_topic"] = ((w or {}).get("primary_topic") or {}).get("display_name", "")

        # Autores
        inst_paper, paises = [], []
        if w and w.get("authorships"):
            r["authors"] = [(a.get("author") or {}).get("display_name", "") for a in w["authorships"]]
            for pos, a in enumerate(w["authorships"], start=1):
                aut = a.get("author") or {}
                insts = a.get("institutions") or []
                fuente_inst = "openalex"
                if not insts:
                    crudas = [x for x in (a.get("raw_affiliation_strings") or []) if x]
                    insts = [{"display_name": crudas[0][:150]}] if crudas else [{}]
                    fuente_inst = "openalex_raw" if crudas else ""
                for inst in insts:
                    autorias.append({
                        "paper_id": r["paper_id"], "author_order": pos,
                        "author_name": aut.get("display_name", ""),
                        "author_openalex_id": sin_prefijo(aut.get("id"), "https://openalex.org/"),
                        "orcid": sin_prefijo(aut.get("orcid"), "https://orcid.org/"),
                        "institution": inst.get("display_name", "") or "",
                        "institution_openalex_id": sin_prefijo(inst.get("id"), "https://openalex.org/"),
                        "ror": inst.get("ror", "") or "",
                        "country": inst.get("country_code", "") or "",
                        "institution_type": inst.get("type", "") or "",
                        "institution_source": fuente_inst,
                    })
                    if inst.get("display_name"):
                        inst_paper.append(inst["display_name"])
                    if inst.get("country_code"):
                        paises.append(inst["country_code"])
        else:
            r["authors"] = r.get("authors_repo", [])
            for pos, nombre in enumerate(r.get("authors_repo", []), start=1):
                autorias.append({
                    "paper_id": r["paper_id"], "author_order": pos,
                    "author_name": nombre,
                    "author_openalex_id": "", "orcid": "",
                    # Sin datos reales de afiliación: no se inventan (se extraerán del PDF)
                    "institution": "", "institution_openalex_id": "", "ror": "",
                    "country": "", "institution_type": "", "institution_source": "",
                })
        r["institutions"] = list(dict.fromkeys(inst_paper))
        r["countries"] = list(dict.fromkeys(paises))
    return registros, pd.DataFrame(autorias)


# ======================================================================
# 3. Clasificación de alcance (adaptada a glaciares)
# ======================================================================
PAT_GLACIAR = re.compile(
    r"\b(glaciar\w*|glacier\w*|glaciol\w*|ice|nieve|snow|nevado\w*|cordillera\w*|hielo\w*|criosf\w*|cryospher\w*)\b", re.I)
PAT_ECOSISTEMA = re.compile(
    r"\b(ecosist[eé]m\w*|ecosystem\w*|bofedal\w*|humedal\w*|laguna\w*|lago\w*|p[aá]ramo\w*|puna|monta[nñ]a\w*|"
    r"mountain\w*|wetland\w*|highland\w*|altoandin\w*|andes|andin\w*)\b", re.I)
PAT_INVENTARIO = re.compile(
    r"\b(inventario|inventory|catálogo|catalogue|catalog|censo|registro nacional)\b", re.I)

MECANISMOS_REGEX = {
    "retroceso_glaciar": re.compile(
        r"\b(retroceso|retreat\w*|deglacia\w*|desglacia\w*|pérdida|loss|reducci[oó]n|shrink\w*|"
        r"balance de masa|mass balance|derretimiento|deshielo|melt\w*|ablaci[oó]n|ablation|"
        r"cambio clim[aá]tico|climate change|fluctuaci\w*)\b", re.I),
    "peligros": re.compile(
        r"\b(peligro\w*|hazard\w*|aluvi[oó]n\w*|aluviones|GLOF\w*|desborde\w*|avalanch\w*|riesgo\w*|risk\w*|"
        r"inundaci\w*|flood\w*|deslizamiento\w*|landslide\w*|huayco\w*|morrena\w*|moraine\w*|"
        r"alerta temprana|early warning|vulnerabilidad|desastre\w*)\b", re.I),
    "hidrologia": re.compile(
        r"\b(h[ií]dric\w*|hidrol[oó]g\w*|hydrolog\w*|agua|water|caudal\w*|escorrent\w*|runoff|cuenca\w*|"
        r"basin\w*|precipitaci\w*|precipitation|recursos? h[ií]dric\w*|disponibilidad h[ií]dric\w*|"
        r"discharge|r[ií]o\w*|river\w*|manantial\w*|acu[ií]fero\w*)\b", re.I),
    "monitoreo": re.compile(
        r"\b(monitoreo\w*|monitor\w*|inventario\w*|inventor\w*|teledetecci[oó]n|remote sensing|sat[eé]lit\w*|"
        r"SIG|GIS|fotogrametr\w*|photogrammetr\w*|geod[eé]si\w*|geodetic|Landsat|Sentinel|drone\w*|"
        r"LiDAR|batimetr\w*|bathymetr\w*|estaci[oó]n\w* meteorol[oó]gic\w*|medici[oó]n\w*)\b", re.I),
    "ecosistemas": re.compile(
        r"\b(ecosist[eé]m\w*|ecosystem\w*|biodiversidad|biodiversity|flora|fauna|bofedal\w*|humedal\w*|wetland\w*|"
        r"suelo\w*|soil\w*|vegetaci[oó]n|vegetation|p[aá]ramo\w*|puna|pastizal\w*|grassland\w*|turber\w*|"
        r"peat\w*|especie\w*|species|servicios? ecosist[eé]mic\w*|carbono)\b", re.I),
}

# Prototipos semánticos DENTRO del alcance
PROTOTIPOS_DENTRO = {
    "retroceso_glaciar": (
        "Retroceso glaciar y cambios en el balance de masa de los glaciares tropicales de los Andes por el "
        "cambio climático: pérdida de superficie y volumen de hielo, deshielo, deglaciación y fluctuaciones "
        "glaciares. Glacier retreat and mass balance."
    ),
    "peligros": (
        "Peligros de origen glaciar: aluviones, desborde de lagunas glaciares (GLOF), avalanchas, flujos de "
        "detritos, lagunas represadas por morrenas, evaluación de riesgo, vulnerabilidad y sistemas de alerta "
        "temprana en zonas de montaña. Glacial hazards and risk."
    ),
    "hidrologia": (
        "Recursos hídricos de origen glaciar: hidrología de cuencas, caudales de ríos alimentados por glaciares, "
        "escorrentía, precipitación, disponibilidad y seguridad hídrica para las poblaciones aguas abajo. "
        "Water resources and basin hydrology."
    ),
    "monitoreo": (
        "Monitoreo e inventario de glaciares y lagunas: teledetección, imágenes satelitales Landsat y Sentinel, "
        "fotogrametría, drones, batimetría, métodos geodésicos, SIG, estaciones de medición e inventarios "
        "nacionales. Glacier monitoring and inventories."
    ),
    "ecosistemas": (
        "Ecosistemas de montaña: bofedales y humedales altoandinos, puna, pastizales, suelos, vegetación, "
        "biodiversidad, carbono y servicios ecosistémicos en los Andes. Mountain ecosystems."
    ),
}

# Prototipos FUERA del alcance (para detectar documentos no relevantes)
PROTOTIPOS_FUERA = {
    "urbano": "Planificación urbana, infraestructura de ciudades, transporte y gestión municipal.",
    "agricultura_baja": "Agricultura de costa y selva baja, cultivos tropicales sin relación con la montaña.",
    "mineria": "Operaciones mineras, extracción de minerales y procesos metalúrgicos.",
    "marino": "Biología marina, oceanografía y ecosistemas costeros y marinos.",
    "administrativo": ("Documentos administrativos e institucionales: memorias anuales, planes operativos, "
                       "presupuesto, recursos humanos, normas internas y procedimientos burocráticos."),
}


class CodificadorTexto:
    """Embeddings con sentence-transformers; si no está disponible, TF-IDF."""

    def __init__(self, modelo="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", usar_embeddings=True):
        self.modelo, self.metodo = None, "tfidf"
        if usar_embeddings:
            try:
                from sentence_transformers import SentenceTransformer
                self.modelo = SentenceTransformer(modelo)
                self.metodo = f"embeddings ({modelo})"
            except Exception as e:
                log(f"   ⚠️ No se pudo cargar el modelo de embeddings ({e}). Se usa TF-IDF.")

    def codificar(self, textos):
        if self.modelo is not None:
            return np.asarray(self.modelo.encode(
                textos, batch_size=64, normalize_embeddings=True,
                show_progress_bar=len(textos) > 200))
        from sklearn.feature_extraction.text import TfidfVectorizer
        X = TfidfVectorizer(
            stop_words=None, sublinear_tf=True, strip_accents="unicode",
            ngram_range=(1, 2), min_df=1
        ).fit_transform(textos)
        X = X.toarray()
        n = np.linalg.norm(X, axis=1, keepdims=True)
        return X / np.where(n == 0, 1, n)


def clasificar_alcance(registros, codificador, margen_excluir=0.04, margen_revisar=0.0):
    """
    Para cada documento calcula:
      relevancia       0 si no habla de glaciares/ecosistemas; si sí, 1 + nº de ejes detectados
      mecanismos       ejes detectados por reglas
      eje_principal    eje más parecido semánticamente
      sim_alcance      similitud con el alcance (máx. de los prototipos 'dentro')
      sim_fuera        similitud con el dominio excluido más parecido
      sim_semillas     similitud con el promedio de las semillas
      exclusion        '' | 'revisar' | 'excluir'
    Las semillas nunca se excluyen automáticamente.
    """
    textos = [f"{r['title']}. {r.get('abstract', '')}" for r in registros]
    # Si los textos están en español, agregar subjects para mejorar la clasificación
    textos_ext = [
        f"{r['title']}. {r.get('abstract', '')}. {' '.join(r.get('subjects', []))}"
        for r in registros
    ]
    nd, nf = list(PROTOTIPOS_DENTRO), list(PROTOTIPOS_FUERA)
    V = codificador.codificar(
        textos_ext + [PROTOTIPOS_DENTRO[k] for k in nd] + [PROTOTIPOS_FUERA[k] for k in nf]
    )
    D = V[:len(textos_ext)]
    PD = V[len(textos_ext):len(textos_ext) + len(nd)]
    PF = V[len(textos_ext) + len(nd):]
    SD, SF = D @ PD.T, D @ PF.T

    semillas = [i for i, r in enumerate(registros) if r.get("semilla")]
    if semillas:
        c = D[semillas].mean(axis=0)
        c = c / (np.linalg.norm(c) or 1)
        s_sem = D @ c
    else:
        s_sem = np.zeros(len(textos_ext))

    ejes5 = list(range(len(nd)))  # todos los ejes

    for i, r in enumerate(registros):
        texto = textos[i]
        es_glaciar_eco = bool(PAT_GLACIAR.search(texto) or PAT_ECOSISTEMA.search(texto))
        mec = [m for m, p in MECANISMOS_REGEX.items() if p.search(texto)]
        r["mecanismos"] = mec
        r["relevancia"] = (1 + len(mec)) if es_glaciar_eco else (1 if mec else 0)
        r["eje_principal"] = nd[int(np.argmax(SD[i]))]
        r["sim_alcance"] = float(SD[i].max())
        j = int(np.argmax(SF[i]))
        r["sim_fuera"], r["dominio_fuera"] = float(SF[i, j]), nf[j]
        r["sim_semillas"] = float(s_sem[i])
        r["es_inventario"] = bool(PAT_INVENTARIO.search(r["title"]))
        r.setdefault("semilla", False)

        exclusion, motivo = "", ""
        diferencia = r["sim_fuera"] - r["sim_alcance"]
        if diferencia > margen_excluir and not es_glaciar_eco and not mec:
            exclusion, motivo = "excluir", f"semántico: {r['dominio_fuera']} ({diferencia:+.2f})"
        elif diferencia > margen_revisar and not es_glaciar_eco:
            exclusion, motivo = "revisar", f"semántico: {r['dominio_fuera']} ({diferencia:+.2f})"

        if r["semilla"] and exclusion == "excluir":
            exclusion, motivo = "", ""
        r["exclusion"], r["motivo_exclusion"] = exclusion, motivo
    return registros


# ======================================================================
# 4. Decisiones manuales persistentes
# ======================================================================
COLS_DEC = ["paper_id", "decision", "motivo", "revisor", "fecha"]


def cargar_decisiones(ruta):
    if os.path.exists(ruta):
        d = pd.read_csv(ruta, dtype=str).fillna("")
        return d[[c for c in COLS_DEC if c in d.columns]]
    return pd.DataFrame(columns=COLS_DEC)


def registrar_decisiones(ruta, excluir=(), incluir=(), motivo="", revisor=""):
    d = cargar_decisiones(ruta)
    nuevas = [
        {"paper_id": str(p).strip(), "decision": dec, "motivo": motivo,
         "revisor": revisor, "fecha": ahora()}
        for dec, ids in (("excluir", excluir), ("incluir", incluir))
        for p in ids if str(p).strip()
    ]
    if nuevas:
        d = pd.concat([d, pd.DataFrame(nuevas)], ignore_index=True)
        d.to_csv(ruta, index=False, encoding="utf-8")
    return d


def aplicar_decisiones(df, decisiones):
    df = df.copy()
    df["decision_manual"] = ""
    if len(decisiones):
        ultima = decisiones.drop_duplicates("paper_id", keep="last").set_index("paper_id")
        for pid, fila in ultima.iterrows():
            m = df["paper_id"] == pid
            if not m.any():
                continue
            df.loc[m, "decision_manual"] = fila["decision"]
            if fila["decision"] == "excluir":
                df.loc[m, ["exclusion", "motivo_exclusion"]] = [
                    "excluir", f"manual: {fila['motivo']}".strip()]
            elif fila["decision"] == "incluir":
                df.loc[m, ["exclusion", "motivo_exclusion"]] = ["", ""]
    df["forzado"] = (
        (df["semilla"] | (df["decision_manual"] == "incluir"))
        & (df["decision_manual"] != "excluir")
    )
    return df


def aplicar_correcciones(df, correcciones=None, tipos_excluidos=()):
    """Corrige metadatos a mano (p. ej. {'20.500.12748/670': {'year': 2022}}) y excluye
    tipos de documento que no aportan al grafo (p. ej. 'Video')."""
    df = df.copy()
    for pid, cambios in (correcciones or {}).items():
        m = df["paper_id"] == pid
        if not m.any():
            log(f"   ⚠️ Corrección para {pid}: no está entre los candidatos")
            continue
        for col, val in cambios.items():
            df.loc[m, col] = val
            if col == "year":
                df.loc[m, "year_fuente"] = "correccion_manual"
    if tipos_excluidos:
        tipos = {t.lower() for t in tipos_excluidos}
        m = df["doc_type"].str.lower().isin(tipos) & ~df["semilla"]
        df.loc[m, ["exclusion", "motivo_exclusion"]] = ["excluir", "tipo de documento excluido"]
    return df


# ======================================================================
# 5. Selección balanceada
# ======================================================================
def puntaje_seleccion(df):
    """Combina citas, similitud con el alcance, similitud con semillas y relevancia."""
    pct_citas = df.groupby(df["year"].fillna(0))["cited_by_count"].rank(pct=True)
    return (
        0.30 * pct_citas
        + 0.30 * df["sim_alcance"].rank(pct=True)
        + 0.25 * df["sim_semillas"].rank(pct=True)
        + 0.15 * df["relevancia"].rank(pct=True)
    ).round(4)


def seleccionar_corpus(df, tamano, min_relevancia=1, prop_inventarios=0.15, solo_con_pdf=True, max_por_serie=4):
    df = df.copy()
    df["puntaje"] = puntaje_seleccion(df)
    elegible = (df["relevancia"] >= min_relevancia) & (df["exclusion"] != "excluir")
    if solo_con_pdf and "pdf_estado" in df.columns:
        # Sin PDF descargable la Persona 2 no puede extraer el texto
        elegible &= df["pdf_estado"].eq("disponible")
    elegidos = df[df["forzado"]].sort_values("puntaje", ascending=False)["paper_id"].tolist()

    serie_de = dict(zip(df["paper_id"], df["serie"] if "serie" in df.columns else [""] * len(df)))
    por_serie = {}
    for pid in elegidos:
        por_serie[serie_de.get(pid, "")] = por_serie.get(serie_de.get(pid, ""), 0) + 1

    def cabe_en_serie(pid):
        sr = serie_de.get(pid, "")
        return not (max_por_serie and sr and por_serie.get(sr, 0) >= max_por_serie)

    # Algunos inventarios/informes técnicos (hubs en el grafo), respetando el tope por serie
    n_inv = int(round(tamano * prop_inventarios)) - int(
        df[df["paper_id"].isin(elegidos)]["es_inventario"].sum())
    if n_inv > 0:
        cand_inv = df[elegible & df["es_inventario"] & ~df["paper_id"].isin(elegidos)].sort_values("puntaje", ascending=False)
        for pid in cand_inv["paper_id"]:
            if n_inv <= 0:
                break
            if cabe_en_serie(pid):
                elegidos.append(pid)
                sr = serie_de.get(pid, "")
                por_serie[sr] = por_serie.get(sr, 0) + 1
                n_inv -= 1

    # Ronda por celdas (año x eje principal) para equilibrar el corpus
    max_inv = max(int(round(tamano * prop_inventarios * 2)), 1)
    n_inv_actual = int(df[df["paper_id"].isin(elegidos)]["es_inventario"].sum())
    cand = df[elegible & ~df["paper_id"].isin(elegidos)].sort_values(
        "puntaje", ascending=False)
    es_inv = dict(zip(cand["paper_id"], cand["es_inventario"]))
    colas = {
        k: list(g["paper_id"])
        for k, g in cand.assign(_y=cand["year"].fillna(0)).groupby(["_y", "eje_principal"], sort=True)
    }
    reserva = []
    while len(elegidos) < tamano and any(colas.values()):
        for k in list(colas):
            while colas[k] and len(elegidos) < tamano:
                pid = colas[k].pop(0)
                if es_inv.get(pid) and n_inv_actual >= max_inv:
                    reserva.append(pid)
                    continue
                if not cabe_en_serie(pid):
                    continue  # evita llenar el corpus con boletines o reportes repetitivos
                sr = serie_de.get(pid, "")
                por_serie[sr] = por_serie.get(sr, 0) + 1
                elegidos.append(pid)
                n_inv_actual += es_inv.get(pid, False)
                break
    elegidos += reserva[:max(tamano - len(elegidos), 0)]
    df["seleccionado"] = df["paper_id"].isin(elegidos)
    return df


# ======================================================================
# Tablas de salida
# ======================================================================
def a_dataframe(registros):
    filas = []
    for r in registros:
        filas.append({
            "paper_id": r.get("paper_id") or r["handle"],
            "handle": r["handle"],
            "item_id": r["item_id"],
            "title": r["title"],
            "year": r.get("year"),
            "year_repo": r.get("year_repo"),
            "year_fuente": r.get("year_fuente", ""),
            "published_date": r["published_date"],
            "doc_type": r.get("doc_type", ""),
            "publisher": r.get("publisher", ""),
            "rights": r.get("rights", ""),
            "pdf_estado": r.get("pdf_estado", ""),
            "pdf_url": r.get("pdf_url", "") or "",
            "pdf_bytes": r.get("pdf_bytes"),
            "community": r.get("community", ""),
            "venue": r.get("venue", ""),
            "venue_fuente": r.get("venue_fuente", ""),
            "fuente": r.get("fuente", "INAIGEM"),
            "source": r.get("fuente", "INAIGEM") + ("+OpenAlex" if r.get("en_openalex") else ""),
            "serie": clave_serie(r["title"]),
            "doi": r.get("doi", ""),
            "openalex_id": r.get("openalex_id", ""),
            "language": r.get("language", ""),
            "subjects": ";".join(r.get("subjects", [])),
            "authors": ";".join(r.get("authors", [])),
            "n_authors": len(r.get("authors", [])),
            "institutions": ";".join(r.get("institutions", [])),
            "countries": ";".join(r.get("countries", [])),
            "openalex_topic": r.get("openalex_topic", ""),
            "cited_by_count": r.get("cited_by_count", 0),
            "cited_by_fuente": r.get("cited_by_fuente", ""),
            "n_referencias": len(r.get("referenced_works", [])),
            "uri": r.get("uri", ""),
            "relevancia": r.get("relevancia", 0),
            "mecanismos": ";".join(r.get("mecanismos", [])),
            "eje_principal": r.get("eje_principal", ""),
            "es_inventario": r.get("es_inventario", False),
            "sim_alcance": round(r.get("sim_alcance", 0), 4),
            "sim_fuera": round(r.get("sim_fuera", 0), 4),
            "dominio_fuera": r.get("dominio_fuera", ""),
            "sim_semillas": round(r.get("sim_semillas", 0), 4),
            "exclusion": r.get("exclusion", ""),
            "motivo_exclusion": r.get("motivo_exclusion", ""),
            "semilla": bool(r.get("semilla", False)),
            "abstract": r.get("abstract", ""),
        })
    return pd.DataFrame(filas)


def citas_internas(registros, ids):
    """Citas internas entre documentos del corpus (via OpenAlex)."""
    oa_a_paper = {
        r.get("openalex_id"): r["paper_id"]
        for r in registros
        if r["paper_id"] in ids and r.get("openalex_id")
    }
    filas = {}
    for r in registros:
        if r["paper_id"] not in ids:
            continue
        for ref in r.get("referenced_works", []):
            d = oa_a_paper.get(ref)
            if d and d != r["paper_id"]:
                filas.setdefault((r["paper_id"], d), set()).add("openalex")
    return pd.DataFrame(
        [{"citing_paper_id": a, "cited_paper_id": b, "source": ";".join(sorted(s))}
         for (a, b), s in sorted(filas.items())],
        columns=["citing_paper_id", "cited_paper_id", "source"]
    )


COLS_TRIPLETAS = [
    "subject", "subject_type", "predicate", "object", "object_type",
    "paper_id", "section", "evidence", "method", "source"
]


def tripletas_metadatos(corpus, autorias, citas):
    """Genera tripletas RDF-like a partir de los metadatos del corpus."""
    t = []
    for f in corpus.itertuples():
        src = "openalex" if f.openalex_id else "inaigem"
        t.append((f.paper_id, "Documento", "tieneTitulo", f.title, "Literal",
                  f.paper_id, "metadata", "", "api", src))
        if pd.notna(f.year) and f.year:
            t.append((f.paper_id, "Documento", "anioPublicacion", str(int(f.year)), "Literal",
                      f.paper_id, "metadata", "", "api", f.year_fuente or src))
        t.append((f.paper_id, "Documento", "publicadoEn", f.venue, "FuentePublicacion",
                  f.paper_id, "metadata", "", "api", f.venue_fuente))
        if getattr(f, "publisher", ""):
            t.append((f.paper_id, "Documento", "publicadoPor", f.publisher, "Institucion",
                      f.paper_id, "metadata", "", "api", "inaigem"))
        if f.doc_type:
            t.append((f.paper_id, "Documento", "tipoDocumento", f.doc_type, "Literal",
                      f.paper_id, "metadata", "", "api", "inaigem"))
        if f.community:
            t.append((f.paper_id, "Documento", "perteneceALinea", f.community, "LineaInvestigacion",
                      f.paper_id, "metadata", "", "api", "inaigem"))
        if f.eje_principal:
            t.append((f.paper_id, "Documento", "tieneEjeTematico", f.eje_principal, "EjeTematico",
                      f.paper_id, "metadata", "", "clasificador", "pipeline"))
        # Subjects como temas
        for subj in (f.subjects or "").split(";"):
            subj = subj.strip()
            if subj:
                t.append((f.paper_id, "Documento", "tieneTema", subj, "Tema",
                          f.paper_id, "metadata", "", "api", "inaigem"))

    # Autorías
    a = autorias.drop_duplicates(["paper_id", "author_name"])
    for x in a.itertuples():
        t.append((x.author_name, "Autor", "autorDe", x.paper_id, "Documento",
                  x.paper_id, "metadata", "", "api",
                  "openalex" if x.author_openalex_id else "inaigem"))

    # Afiliaciones
    b = autorias[
        autorias["institution"] != ""
    ].drop_duplicates(["paper_id", "author_name", "institution"])
    for x in b.itertuples():
        t.append((x.author_name, "Autor", "afiliadoA", x.institution, "Institucion",
                  x.paper_id, "metadata", "", "api", x.institution_source))

    # Países
    c = autorias[
        (autorias["institution"] != "") & (autorias["country"] != "")
    ].drop_duplicates(["institution", "country"])
    for x in c.itertuples():
        t.append((x.institution, "Institucion", "ubicadaEn", x.country, "Pais",
                  x.paper_id, "metadata", "", "api", "openalex"))

    # Citas internas
    for x in citas.itertuples():
        t.append((x.citing_paper_id, "Documento", "cita", x.cited_paper_id, "Documento",
                  x.citing_paper_id, "references", "", "api", x.source))

    return pd.DataFrame(t, columns=COLS_TRIPLETAS)


# ======================================================================
# 6. Reporte de calidad y registro
# ======================================================================
def reporte_calidad(corpus, candidatos, autorias, citas, tamano_objetivo, semillas_cfg, manifiesto=None):
    ok, warn, err = "✅", "⚠️", "❌"
    chk = []
    n = len(corpus)
    chk.append((ok if n >= tamano_objetivo else warn,
                f"Tamaño del corpus: {n} (objetivo {tamano_objetivo})"))
    dup = corpus["paper_id"].duplicated().sum() + corpus["title"].map(titulo_normalizado).duplicated().sum()
    chk.append((ok if dup == 0 else err, f"Duplicados: {dup}"))
    sin_abs = (corpus["abstract"].str.len() < 50).sum()
    chk.append((ok if sin_abs == 0 else warn,
                f"Documentos sin abstract: {sin_abs}" +
                ("  → extraer desde el PDF (etapa PLN)" if sin_abs else "")))
    excl_manual = set(candidatos.loc[candidatos["decision_manual"] == "excluir", "paper_id"])
    handles_semilla = set(semillas_cfg.keys())
    # Verificar semillas por handle
    semillas_en_corpus = set(corpus[corpus["handle"].isin(handles_semilla)]["paper_id"])
    chk.append((ok if len(semillas_en_corpus) == len(handles_semilla) else warn,
                f"Semillas en el corpus: {len(semillas_en_corpus)}/{len(handles_semilla)}"))
    pend = corpus[(corpus["exclusion"] == "revisar") & (corpus["decision_manual"] == "")]
    chk.append((ok if len(pend) == 0 else warn,
                f"Seleccionados pendientes de revisión manual: {len(pend)}"))
    p_oa = corpus["openalex_id"].astype(bool).mean() * 100 if n else 0
    chk.append((ok if p_oa >= 30 else warn,
                f"Con metadatos de OpenAlex: {p_oa:.0f}% (esperado: solo docs con DOI)"))

    if "fuente" in corpus.columns:
        chk.append((ok, "Documentos por fuente: " + ", ".join(f"{k}: {v}" for k, v in corpus["fuente"].value_counts().items())))
    if "serie" in corpus.columns:
        mayor = corpus["serie"].value_counts()
        if len(mayor):
            chk.append((ok if mayor.iloc[0] <= 5 else warn, f"Serie más repetida: {mayor.index[0][:40]}… ({mayor.iloc[0]} docs)"))

    # Distribución por comunidad
    if "community" in corpus.columns:
        base_com = corpus[corpus["fuente"] == "INAIGEM"] if "fuente" in corpus.columns else corpus
        com_dist = base_com["community"].replace("", "(sin línea)").value_counts()
        chk.append((ok, "Líneas de investigación (INAIGEM): " +
                    ", ".join(f"{k}: {v}" for k, v in com_dist.items())))

    malos = corpus["year"].isna() | ~corpus["year"].fillna(0).between(ANIO_MIN, ANIO_MAX)
    chk.append((ok if malos.sum() == 0 else warn, f"Documentos sin año válido: {int(malos.sum())}"
                + (f" → corregir en CORRECCIONES: {corpus.loc[malos, 'paper_id'].tolist()[:5]}" if malos.any() else "")))
    if "year_fuente" in corpus.columns:
        alt = corpus["year_fuente"].isin(["titulo", "dc.date.available", "dc.date.accessioned", "dc.identifier.citation"]).sum()
        if alt:
            chk.append((warn, f"Año tomado de un campo alternativo (verificar): {alt} documentos"))
    if "pdf_estado" in candidatos.columns:
        disp = candidatos["pdf_estado"].value_counts().to_dict()
        chk.append((ok, "PDF en los candidatos: " + ", ".join(f"{k}: {v}" for k, v in disp.items())))
        sin_pdf = corpus[corpus["pdf_estado"] != "disponible"]
        chk.append((ok if len(sin_pdf) == 0 else warn, f"Seleccionados sin PDF descargable: {len(sin_pdf)}"
                    + (f" (semillas: {sin_pdf['paper_id'].tolist()})" if len(sin_pdf) else "")))
    con_inst = (autorias[autorias["paper_id"].isin(corpus["paper_id"])]["institution"] != "").mean() * 100 if len(autorias) else 0
    chk.append((ok if con_inst >= 50 else warn, f"Autorías con institución real: {con_inst:.0f}%"
                + ("  → completar desde la 1.ª página del PDF (Persona 2)" if con_inst < 50 else "")))
    anios = corpus["year"].value_counts(normalize=True)
    chk.append((ok if (anios.max() if len(anios) else 0) <= 0.5 else warn,
                "Distribución por año: " +
                ", ".join(f"{int(a)}: {c}" for a, c in corpus["year"].dropna().value_counts().sort_index().items())))

    for eje in MECANISMOS_REGEX:
        c = corpus["mecanismos"].str.contains(eje).sum()
        chk.append((ok if c >= max(3, n * 0.05) else warn,
                    f"Eje {eje}: {c} docs (reglas) | eje principal en "
                    f"{(corpus['eje_principal'] == eje).sum()}"))

    if manifiesto is not None and len(manifiesto):
        m = manifiesto[manifiesto["paper_id"].isin(corpus["paper_id"])]
        pdf_ok = (m["estado"] == "ok").sum()
        chk.append((ok if pdf_ok == n else warn,
                    f"PDFs descargados y válidos: {pdf_ok}/{n}"))

    lineas = [
        "# Reporte de calidad del corpus — Glaciares y Ecosistemas de Montaña", "",
        f"Generado: {ahora()}", "",
        "| Estado | Verificación |", "|---|---|"
    ] + [f"| {e} | {m} |" for e, m in chk]
    lineas += [
        "", "## Embudo de selección", "",
        f"- Candidatos evaluados: {len(candidatos)}",
        f"- Excluidos (automático + manual): {(candidatos['exclusion'] == 'excluir').sum()}",
        f"- Marcados para revisar: {(candidatos['exclusion'] == 'revisar').sum()}",
        f"- Seleccionados: {n}",
    ]
    return chk, "\n".join(lineas)


def registro_busqueda(ruta, conteos, config, metodo_clasificador, cache):
    datos = {
        "fecha_ejecucion_utc": ahora(),
        "version_pipeline": VERSION,
        "fuentes": {
            "principal": "Repositorio Digital INAIGEM (DSpace 7.6.1)",
            "complementarias": "Repositorio de Recursos Hídricos (ANA) y Repositorio Institucional SENAMHI "
                               "(cosecha automática con filtro temático)",
            "metadatos": "OpenAlex API (docs con DOI)",
        },
        "clasificador_alcance": metodo_clasificador,
        "configuracion": {k: v for k, v in config.items() if "KEY" not in k.upper()},
        "conteos": conteos,
        "cache": cache.resumen(),
        "entorno": {"python": platform.python_version(), "pandas": pd.__version__},
    }
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2, default=str)


DICCIONARIO = """# Diccionario de datos — Corpus Glaciares y Ecosistemas de Montaña (Persona 1)

Corpus sobre glaciares y ecosistemas de montaña en Perú. Fuente principal: Repositorio Digital
INAIGEM (DSpace 7.6.1). Fuentes complementarias: Repositorio Digital de Recursos Hídricos (ANA) y
Repositorio Institucional del SENAMHI, de los que solo se toman documentos que mencionan glaciares,
lagunas de origen glaciar, nevados, aluviones, bofedales o alta montaña. OpenAlex: metadatos de docs con DOI.

## Archivos

| Archivo | Contenido |
|---|---|
| `papers.csv` | Corpus seleccionado (1 fila por documento) |
| `candidatos.csv` | Todos los candidatos evaluados, con clasificación |
| `authorships.csv` | Autoría: documento, autor, orden, institución, país |
| `citations.csv` | Citas entre documentos del corpus (via OpenAlex) |
| `metadata_triples.csv` | Tripletas de metadatos con fuente (entrada al grafo) |
| `decisiones_manuales.csv` | Historial de inclusiones/exclusiones manuales |
| `revision_pendiente.csv` | Seleccionados que requieren revisión humana |
| `pdfs_manifest.csv` | Estado de cada PDF (ok/error), tamaño, páginas y hash |
| `search_log.json` | Configuración, conteos y fecha (reproducibilidad) |
| `reporte_calidad.md` | Verificaciones automáticas del corpus |
| `cache/` | Respuestas de las APIs (permite repetir sin reconsultar) |

## Columnas principales de `papers.csv`

| Columna | Descripción |
|---|---|
| `paper_id` / `handle` | Identificador del documento (handle; el prefijo indica la institución: 20.500.12748 INAIGEM, 20.500.12543 ANA, 20.500.12542 SENAMHI) |
| `fuente` | Repositorio de origen (INAIGEM, ANA, SENAMHI) |
| `serie` | Clave de serie (boletines/reportes numerados); se limita cuántos de una misma serie entran |
| `item_id` | UUID del ítem en DSpace |
| `year`, `year_fuente` | Año de publicación y campo del que se obtuvo (fechas imposibles se descartan) |
| `publisher` | Institución editora del documento (relación `publicadoPor`) |
| `pdf_estado` | `disponible` (acceso abierto), `restringido` (requiere inicio de sesión), `sin_pdf` |
| `doc_type` | Tipo de documento (Report, Journal Article, etc.) |
| `community` | Línea de investigación INAIGEM |
| `venue`, `venue_fuente` | Publicación y de dónde se obtuvo |
| `subjects` | Palabras clave del repositorio (separadas por ;) |
| `authors`, `institutions`, `countries` | Listas separadas por ;. Instituciones y países solo cuando la fuente (OpenAlex) los registra; no se infieren |
| `relevancia` | 0 = no trata de glaciares/ecosistemas; 1+ = sí |
| `mecanismos` | Ejes detectados por reglas |
| `eje_principal` | Eje más parecido semánticamente |
| `es_inventario` | True si es un inventario o catálogo nacional |

## Ejes temáticos

| Eje | Descripción |
|---|---|
| retroceso_glaciar | Retroceso, balance de masa, deglaciación |
| peligros | Peligros glaciares, GLOF, aluviones, riesgo |
| hidrologia | Recursos hídricos, cuencas, escorrentía |
| monitoreo | Inventarios, teledetección, monitoreo |
| ecosistemas | Bofedales, biodiversidad, ecosistemas de montaña |

## Tripletas (`metadata_triples.csv`)

`subject, subject_type, predicate, object, object_type, paper_id, section, evidence, method, source`

Predicados: `tieneTitulo`, `anioPublicacion`, `publicadoEn`, `tipoDocumento`, `perteneceALinea`,
`tieneEjeTematico`, `tieneTema`, `publicadoPor`, `autorDe`, `afiliadoA`, `ubicadaEn`, `cita`.

`afiliadoA` solo se registra cuando OpenAlex tiene la afiliación. Las demás afiliaciones se
extraen de la primera página del PDF (Persona 2). Solo se seleccionan documentos con PDF descargable.
"""


# ======================================================================
# 7. PDFs (descarga desde DSpace)
# ======================================================================
COLS_MANIF = ["paper_id", "archivo", "estado", "bytes", "paginas", "sha256", "error", "fecha"]


def contar_paginas(ruta):
    try:
        try:
            import pymupdf
        except ImportError:
            import fitz as pymupdf
        with pymupdf.open(ruta) as d:
            return d.page_count
    except Exception:
        return None


def obtener_pdf_dspace(base_url, item_uuid, cache, pausa=0.3):
    """Busca en el bundle ORIGINAL el PDF principal (el más grande) y devuelve
    {'url', 'nombre', 'bytes'} o None si el ítem no tiene PDF."""
    clave = f"pdf_info|{item_uuid}"
    if cache.tiene("inaigem_pdf", clave):
        return cache.get("inaigem_pdf", clave)
    info = None
    try:
        r = http("GET", f"{base_url}/core/items/{item_uuid}/bundles", intentos=3, espera=3)
        if r.status_code == 200:
            info = {"url": "", "nombre": "", "bytes": None}  # consultado: sin PDF salvo que se encuentre
            for bundle in r.json().get("_embedded", {}).get("bundles", []):
                if bundle.get("name") != "ORIGINAL":
                    continue
                link = bundle.get("_links", {}).get("bitstreams", {}).get("href", "")
                if not link:
                    continue
                r2 = http("GET", link, intentos=3, espera=3)
                if r2.status_code != 200:
                    continue
                pdfs = [b for b in r2.json().get("_embedded", {}).get("bitstreams", [])
                        if b.get("name", "").lower().endswith(".pdf")]
                if pdfs:
                    b = max(pdfs, key=lambda x: x.get("sizeBytes") or 0)
                    info = {"url": b.get("_links", {}).get("content", {}).get("href", ""),
                            "nombre": b.get("name", ""), "bytes": b.get("sizeBytes")}
    except AccesoDenegado:
        info = {"url": "", "nombre": "", "bytes": None, "restringido_item": True}
    except Exception as e:
        log(f"   ⚠️ {item_uuid}: {str(e)[:120]}")
        return None  # error de red: no se guarda en caché, se reintentará
    cache.put("inaigem_pdf", clave, info)
    time.sleep(pausa)
    return info


def probar_acceso_pdf(url, timeout=60):
    """Pide solo los primeros bytes del PDF para saber si es de acceso abierto."""
    try:
        with _SESION.get(url, headers=dict(HEADERS, Range="bytes=0-1023"), stream=True, timeout=timeout) as r:
            if r.status_code in (401, 403):
                return "restringido"
            if r.status_code in (200, 206):
                inicio = next(r.iter_content(1024), b"")
                return "disponible" if inicio[:4] == b"%PDF" else "no_es_pdf"
            return f"http_{r.status_code}"
    except Exception:
        return "error_red"


def verificar_pdfs(registros, base_url, cache, pausa=0.3):
    """Marca cada candidato con pdf_estado: disponible / restringido / sin_pdf / error.
    Así la selección solo elige documentos cuyo PDF se puede descargar."""
    total = len(registros)
    pendientes = 0
    for i, r in enumerate(registros, start=1):
        clave = f"acceso|{r['item_id']}"
        if cache.tiene("inaigem_pdf", clave):
            r["pdf_estado"], r["pdf_url"], r["pdf_bytes"] = cache.get("inaigem_pdf", clave)
            continue
        pendientes += 1
        metodo = r.get("fuente_metodo", "dspace7")
        if r.get("pdf_url_hint"):
            info = {"url": r["pdf_url_hint"], "nombre": "", "bytes": None}
        elif metodo == "rest6":
            info = {"url": ""}  # la API REST ya listó sus archivos y no había PDF
        elif metodo == "dspace7":
            base = r.get("fuente_api") or base_url
            info = obtener_pdf_dspace(base, r["item_id"], cache, pausa)
        elif r.get("handle"):
            # OAI (SENAMHI, DSpace 6): primero la API REST por handle; si no existe, la página del documento
            info = pdf_desde_rest_handle(r["fuente_url"], r["handle"], cache)
            if info is None:
                info = pdf_desde_pagina(r["fuente_url"], r["handle"], cache)
        else:
            info = {"url": ""}
        if info is None:
            estado, url, nbytes = "error", "", None
        elif info.get("restringido_item"):
            estado, url, nbytes = "restringido", "", None
        elif not info.get("url"):
            estado, url, nbytes = "sin_pdf", "", None
        else:
            estado, url, nbytes = probar_acceso_pdf(info["url"]), info["url"], info.get("bytes")
            time.sleep(pausa)
        r["pdf_estado"], r["pdf_url"], r["pdf_bytes"] = estado, url, nbytes
        if estado in ("disponible", "restringido", "sin_pdf"):  # los errores de red se reintentan luego
            cache.put("inaigem_pdf", clave, [estado, url, nbytes])
        if pendientes % 50 == 0:
            log(f"   {i} / {total} verificados")
    conteo = pd.crosstab(pd.Series([r.get("fuente", "") for r in registros], name="fuente"),
                         pd.Series([r["pdf_estado"] for r in registros], name="pdf"))
    log("PDF por fuente y estado:\n" + conteo.to_string())
    return registros


def descargar_pdfs(corpus, carpeta, ruta_manifiesto, base_url, cache, pausa=2.0):
    os.makedirs(carpeta, exist_ok=True)
    man = (pd.read_csv(ruta_manifiesto, dtype={"paper_id": str})
           if os.path.exists(ruta_manifiesto)
           else pd.DataFrame(columns=COLS_MANIF))
    man = (man.set_index("paper_id") if len(man)
           else pd.DataFrame(columns=COLS_MANIF).set_index("paper_id"))
    total = len(corpus)
    for i, f in enumerate(corpus.itertuples(), start=1):
        base_nombre = f.handle if isinstance(f.handle, str) and f.handle else str(f.item_id)
        nombre = re.sub(r"[^\w.-]", "_", base_nombre) + ".pdf"   # válido en Windows (sin / : ? *)
        destino = os.path.join(carpeta, nombre)
        if os.path.exists(destino) and os.path.getsize(destino) > 1000:
            if f.paper_id not in man.index or man.loc[f.paper_id, "estado"] != "ok":
                with open(destino, "rb") as fh:
                    contenido = fh.read()
                man.loc[f.paper_id] = [
                    nombre, "ok", len(contenido), contar_paginas(destino),
                    hashlib.sha256(contenido).hexdigest(), "", ahora()
                ]
            continue
        error = ""
        pdf_url = getattr(f, "pdf_url", "") or ""
        if not pdf_url:
            info = obtener_pdf_dspace(base_url, f.item_id, cache)
            pdf_url = (info or {}).get("url", "")
        if pdf_url:
            try:
                r = http("GET", pdf_url, intentos=3, espera=5, timeout=120)
                if r.status_code == 200 and r.content[:4] == b"%PDF":
                    with open(destino, "wb") as fh:
                        fh.write(r.content)
                    man.loc[f.paper_id] = [
                        nombre, "ok", len(r.content), contar_paginas(destino),
                        hashlib.sha256(r.content).hexdigest(), "", ahora()
                    ]
                    error = ""
                else:
                    error = f"HTTP {r.status_code} o contenido no PDF"
            except Exception as e:
                error = str(e)[:200]
        else:
            error = "No se encontró bitstream PDF en DSpace"
        if error:
            man.loc[f.paper_id] = [nombre, "error", 0, None, "", error, ahora()]
            log(f"   ❌ {f.paper_id}: {error}")
        if i % 10 == 0 or i == total:
            log(f"   {i} / {total} procesados")
            man.reset_index().to_csv(ruta_manifiesto, index=False)
        time.sleep(pausa)
    man = man.reset_index()
    man.to_csv(ruta_manifiesto, index=False)
    ok = man[man["paper_id"].isin(corpus["paper_id"]) & (man["estado"] == "ok")]
    log(f"PDFs válidos: {len(ok)} / {total}")
    return man


# ======================================================================
# 8. v5 — Decisiones declaradas en el notebook (una sola fuente de verdad)
# ======================================================================
def construir_decisiones(df_base, reglas_titulo=None, excluir=None, incluir=None, revisor=""):
    """Arma la tabla de decisiones a partir de lo escrito en el notebook.
    Orden: reglas por título → EXCLUIR → INCLUIR. Como vale la última decisión de cada
    documento, INCLUIR siempre gana. Las semillas no se ven afectadas por las reglas."""
    filas, fecha = [], ahora()
    resumen = []
    for motivo, patron in (reglas_titulo or {}).items():
        m = df_base["title"].str.contains(patron, case=False, regex=True, na=False) & ~df_base["semilla"]
        ids = df_base.loc[m, "paper_id"].tolist()
        resumen.append(("regla", motivo, len(ids)))
        filas += [(p, "excluir", motivo) for p in ids]
    for decision, grupos in (("excluir", excluir or {}), ("incluir", incluir or {})):
        for motivo, ids in grupos.items():
            resumen.append((decision, motivo, len(ids)))
            filas += [(str(p).strip(), decision, motivo) for p in ids if str(p).strip()]
    d = pd.DataFrame(filas, columns=["paper_id", "decision", "motivo"])
    d["revisor"], d["fecha"] = revisor, fecha
    no_estan = sorted(set(d["paper_id"]) - set(df_base["paper_id"]))
    return d[COLS_DEC], pd.DataFrame(resumen, columns=["tipo", "motivo", "documentos"]), no_estan


def guardar_decisiones(decisiones, ruta):
    """Guarda las decisiones declaradas en el notebook. La primera vez respalda el archivo
    anterior (decisiones registradas celda por celda en la v4) como *_historial_v4.csv."""
    respaldo = ruta.replace(".csv", "_historial_v4.csv")
    if os.path.exists(ruta) and not os.path.exists(respaldo):
        cargar_decisiones(ruta).to_csv(respaldo, index=False, encoding="utf-8")
        log(f"Respaldo del archivo anterior: {os.path.basename(respaldo)}")
    decisiones.to_csv(ruta, index=False, encoding="utf-8")
    return respaldo if os.path.exists(respaldo) else None


def guardar_resultados(carpeta, corpus, df, registros, autorias, total_repo, resumen_fuentes,
                       config, semillas, metodo_clasificador, cache, ruta_decisiones):
    """Sección 8: escribe todos los archivos de salida y el reporte de calidad."""
    ids = set(corpus["paper_id"])
    aut_corpus = autorias[autorias["paper_id"].isin(ids)]
    citas = citas_internas(registros, ids)
    trip = tripletas_metadatos(corpus, aut_corpus, citas)
    pendientes = corpus[(corpus["exclusion"] == "revisar") & (corpus["decision_manual"] == "")]
    ruta_manif = os.path.join(carpeta, "pdfs_manifest.csv")
    manif = pd.read_csv(ruta_manif, dtype={"paper_id": str}) if os.path.exists(ruta_manif) else None

    def a_csv(tabla, nombre):
        tabla.to_csv(os.path.join(carpeta, nombre), index=False, encoding="utf-8")

    a_csv(corpus.drop(columns=["seleccionado"]), "papers.csv")
    a_csv(df, "candidatos.csv")
    a_csv(aut_corpus, "authorships.csv")
    a_csv(citas, "citations.csv")
    a_csv(trip, "metadata_triples.csv")
    a_csv(pendientes[["paper_id", "title", "motivo_exclusion"]], "revision_pendiente.csv")
    with open(os.path.join(carpeta, "README_datos.md"), "w", encoding="utf-8") as f:
        f.write(DICCIONARIO)

    conteos = {
        "total_repositorio": total_repo,
        "fuentes": resumen_fuentes,
        "candidatos_descargados": len(registros),
        "con_openalex": sum(r["en_openalex"] for r in registros),
        "relevantes": int((df["relevancia"] >= config["MIN_RELEVANCIA"]).sum()),
        "excluidos": int((df["exclusion"] == "excluir").sum()),
        "para_revisar": int((df["exclusion"] == "revisar").sum()),
        "decisiones_manuales": len(cargar_decisiones(ruta_decisiones)),
        "seleccionados": len(corpus),
        "autorias": len(aut_corpus),
        "citas_internas": len(citas),
        "tripletas_metadatos": len(trip),
    }
    registro_busqueda(os.path.join(carpeta, "search_log.json"), conteos, config, metodo_clasificador, cache)
    checks, reporte = reporte_calidad(corpus, df, autorias, citas, config["TAMANO_CORPUS"], semillas, manif)
    with open(os.path.join(carpeta, "reporte_calidad.md"), "w", encoding="utf-8") as f:
        f.write(reporte)
    return {"checks": checks, "conteos": conteos, "autorias": aut_corpus, "citas": citas,
            "tripletas": trip, "pendientes": pendientes}


def pdfs_fuera_del_corpus(carpeta_pdfs, corpus, mover=False):
    """PDFs descargados antes que ya no están en el corpus (por decisiones posteriores).
    Con mover=True se trasladan a pdfs/_fuera_del_corpus (no se borran)."""
    if not os.path.isdir(carpeta_pdfs):
        return []
    validos = {re.sub(r"[^\w.-]", "_", (h if isinstance(h, str) and h else str(i))) + ".pdf"
               for h, i in zip(corpus["handle"], corpus["item_id"])}
    sobran = sorted(f for f in os.listdir(carpeta_pdfs)
                    if f.lower().endswith(".pdf") and f not in validos)
    if mover and sobran:
        destino = os.path.join(carpeta_pdfs, "_fuera_del_corpus")
        os.makedirs(destino, exist_ok=True)
        for f in sobran:
            os.replace(os.path.join(carpeta_pdfs, f), os.path.join(destino, f))
    return sobran


print("Pipeline cargado · versión", VERSION)

## 3b. Diagnóstico de conexión
Prueba en segundos si cada servidor responde. ✅ responde · ⚠️ responde con error · ❌ sin conexión.
Si una fuente sale ❌, el notebook continúa sin ella (lo ya descargado sigue en la caché).

In [ ]:
import time

def _url_prueba(cfg):
    if cfg.get("metodo") == "dspace7":
        return (cfg.get("api") or cfg["url"].rstrip("/") + "/server/api") + "/core/sites"
    if cfg.get("metodo") == "oai":
        return cfg["url"].rstrip("/") + "/oai/request?verb=Identify"
    return cfg["url"]

PRUEBAS = {"INAIGEM": CONFIG["REPO_BASE_URL"] + "/core/sites"}
PRUEBAS.update({n: _url_prueba(c) for n, c in CONFIG["FUENTES_ADICIONALES"].items()})
PRUEBAS["OpenAlex"] = "https://api.openalex.org/works?per_page=1"
PRUEBAS["Hugging Face (modelo)"] = "https://huggingface.co/api/models/" + CONFIG["MODELO_EMBEDDINGS"]

CONEXION = {}
for nombre, url in PRUEBAS.items():
    t0 = time.time()
    try:
        r = _SESION.get(url, headers=HEADERS, timeout=(20, 60))
        ok = r.status_code < 400
        icono, detalle = ("✅" if ok else "⚠️"), f"HTTP {r.status_code}"
    except Exception as e:
        ok, icono, detalle = False, "❌", e.__class__.__name__
    CONEXION[nombre] = ok
    print(f"{icono} {nombre:<24} {detalle:<22} {time.time() - t0:5.1f} s")

caidas = [n for n in CONFIG["FUENTES_ADICIONALES"] if not CONEXION.get(n)]
if caidas:
    print(f"\n⚠️ No responde(n): {', '.join(caidas)}. "
          + ("Desde Colab es normal: usar Jupyter en la computadora." if EN_COLAB
             else "Revisar VPN/proxy o probar más tarde; se continúa con la caché."))
else:
    print("\nTodo listo.")

## 4. Cosecha de repositorios (INAIGEM, ANA, SENAMHI)
INAIGEM se descarga completo. De la ANA y el SENAMHI solo se conservan los documentos del tema
(glaciares, lagunas de origen glaciar, nevados, aluviones, bofedales, alta montaña). Todo queda en caché.

In [ ]:
from collections import Counter

cache = Cache(os.path.join(CARPETA, "cache"), activa=CONFIG["USAR_CACHE"])
RUTA_DECISIONES = os.path.join(CARPETA, "decisiones_manuales.csv")

print("Repositorio INAIGEM...")
registros, total_repo = buscar_repositorio_inaigem(CONFIG["REPO_BASE_URL"], cache, max_items=CONFIG["MAX_CANDIDATOS"])
registros = asignar_comunidades(registros, CONFIG["REPO_BASE_URL"], CONFIG["COMUNIDADES"], cache)
registros = buscar_semillas_inaigem(SEMILLAS_HANDLES, registros)

resumen_fuentes = {"INAIGEM": {"metodo": "dspace7", "cosechados": len(registros), "del_tema": len(registros)}}
for nombre, cfg_f in CONFIG["FUENTES_ADICIONALES"].items():
    print(f"\n{nombre}...")
    regs_f, info_f = cosechar_fuente(nombre, cfg_f, cache)
    for r in regs_f:
        r.setdefault("semilla", False)
    registros += regs_f
    resumen_fuentes[nombre] = info_f

n0 = len(registros)
registros = deduplicar(registros)   # si un documento está en dos repositorios, queda la versión del INAIGEM

print("\n" + "=" * 60)
print(pd.DataFrame(resumen_fuentes).T.to_string())
print(f"\nDuplicados entre fuentes eliminados: {n0 - len(registros)}")
print("Candidatos por fuente:", dict(Counter(r.get("fuente", "INAIGEM") for r in registros)))
print("Semillas encontradas:", [r["handle"] for r in registros if r.get("semilla")])

## 5. Enriquecimiento con OpenAlex (solo documentos con DOI)

In [ ]:
works_oa = {}
if CONFIG["USAR_OPENALEX"]:
    dois = [r["doi"] for r in registros if r.get("doi")]
    works_oa = consultar_openalex(dois, OPENALEX_API_KEY, cache)

registros, autorias = integrar_metadatos(registros, works_oa)
print(f"Enriquecidos con OpenAlex: {sum(r['en_openalex'] for r in registros)}/{len(registros)}")
print(f"Autorías registradas: {len(autorias)}")

## 5b. Verificar qué PDFs se pueden descargar
Se piden solo los primeros bytes de cada PDF: **disponible** (acceso abierto), **restringido** (pide iniciar sesión), **sin_pdf**.

In [ ]:
registros = verificar_pdfs(registros, CONFIG["REPO_BASE_URL"], cache)
estados = pd.Series([r["pdf_estado"] for r in registros])
print("Estado de los PDF:", estados.value_counts().to_dict())

## 6. Clasificación de alcance
- **Reglas:** palabras clave por eje (retroceso glaciar, peligros, hidrología, monitoreo, ecosistemas).
- **Semántica:** similitud con descripciones en español de cada eje (modelo multilingüe).
- Las **semillas** nunca se excluyen automáticamente.

In [ ]:
codificador = CodificadorTexto(CONFIG["MODELO_EMBEDDINGS"], CONFIG["USAR_EMBEDDINGS"])
print("Clasificador:", codificador.metodo)
registros = clasificar_alcance(registros, codificador, CONFIG["MARGEN_EXCLUIR"], CONFIG["MARGEN_REVISAR"])
df_base = a_dataframe(registros)

auto = df_base[df_base.exclusion == "excluir"]
print(f"Candidatos: {len(df_base)} | excluidos por el clasificador: {len(auto)} | "
      f"marcados para revisar: {(df_base.exclusion == 'revisar').sum()}")
if len(auto):
    print("\nMuestra de excluidos automáticamente (control de calidad):")
    display(auto.sample(min(10, len(auto)), random_state=1)[["paper_id", "title", "fuente", "motivo_exclusion"]])

## 7. Decisiones manuales ✏️ (única celda de decisiones)
Aquí está **todo** lo que se decidió en la revisión, agrupado por motivo. Se aplica en este orden:
1. **Reglas por título** (`REGLAS_TITULO`): excluyen familias completas (boletines, pronósticos, Antártida…) y evitan que entren reemplazos parecidos.
2. **`EXCLUIR`**: documentos puntuales.
3. **`INCLUIR`**: documentos que entran sí o sí. **Gana sobre todo lo anterior**.

Para cambiar algo, se edita esta celda y se vuelve a ejecutar desde aquí hacia abajo. El archivo `decisiones_manuales.csv` se reescribe a partir de esta celda (la primera vez se guarda una copia del anterior como `decisiones_manuales_historial_v4.csv`).

In [ ]:
REVISOR = "G8 - Persona 1"

# ── 1. Reglas por título (expresiones regulares; no afectan a las semillas) ──
REGLAS_TITULO = {
    "boletín periódico u operativo sin contenido glaciar":
        r"bolet[ií]n (?:regional|agroclim\w*|semanal de lluvia|meteorol[oó]gico e hidrol\w*|agro-?hidro\w*)"
        r"|informaci[oó]n hidrol[oó]gica diaria",
    "pronóstico o producto climático operativo (sin contenido glaciar)":
        r"perspectivas clim\w*|perspectivas para|riesgo agroclim\w*|pron[oó]stico de riesgo"
        r"|pron[oó]stico estacional|bolet[ií]n clim[aá]tico|vigilancia sin[oó]ptica",
    "fuera de alcance: Antártida (el alcance es el Perú)":
        r"ant[aá]rti\w*|king george|admiralty bay|maxwell bay|znosko|shetland",
    "historieta, cartilla o material infantil con poco texto para PLN":
        r"historieta|cartilla|para colorear",
}

# ── 2. Exclusiones puntuales ──
EXCLUIR = {
    "revisión manual: no es documento técnico del dominio": [
        "20.500.12748/123", "20.500.12748/677", "20.500.12748/610",
    ],
    "fuera de alcance: laguna costera o valle sin glaciares": [
        "20.500.12543/2464",  # Lagunas de Mejía (costa de Arequipa)
        "20.500.12543/2077", "20.500.12543/2055",  # Laguna de Huacachina, Ica
        "20.500.12543/2945",  # Riego del valle del río Zaña
        "20.500.12543/3280",  # Irrigación Las Bayas, pampa de Siguas
    ],
    "fuera de alcance: agricultura y suelos agrícolas": [
        "20.500.12542/2155", "20.500.12542/4988", "20.500.12542/5432",
        "20.500.12542/3247", "20.500.12542/6165",
        "20.500.12543/1012", "20.500.12543/1011",  # Estudios de suelos (Huaylas, Conchucos)
    ],
    "fuera de alcance: Antártida (el alcance es el Perú)": [
        "20.500.12542/1900", "20.500.12542/1854",  # Glaciar Znosko
        "20.500.12748/680",                        # Fitoplancton en Admiralty Bay
    ],
    "boletín periódico u operativo sin contenido glaciar": [   # ya cubiertos por la regla; se dejan explícitos
        "20.500.12542/6108", "20.500.12542/6091", "20.500.12542/6095", "20.500.12542/6103",
        "20.500.12542/4625", "20.500.12542/4647", "20.500.12542/1731", "20.500.12542/2161",
        "20.500.12542/6220",
    ],
    "pronóstico o producto climático operativo (sin contenido glaciar)": [
        "20.500.12542/800", "20.500.12542/2523", "20.500.12542/2271",
    ],
    "infografía, folleto o material infantil con poco texto para PLN": [
        "20.500.12543/5937", "20.500.12543/5910", "20.500.12543/5945", "20.500.12543/5944",
        "20.500.12543/5943", "20.500.12543/5942", "20.500.12543/5941", "20.500.12543/5940",
        "20.500.12543/5912", "20.500.12543/5929", "20.500.12543/2662",  # infografías ANA
        "20.500.12543/2671",  # Conociendo los glaciares (cuatríptico)
        "20.500.12542/2163",  # Infografía clima del Mantaro
        "20.500.12543/5850",  # Cartilla para colorear
        "20.500.12543/2646",  # Los hijos de Mamacocha
        "20.500.12542/3869",  # Historieta sobre retroceso glaciar
    ],
    "evento o capacitación general, fuera del dominio": [
        "20.500.12543/639",   # VI Foro Mundial del Agua
        "20.500.12543/1026",  # Curso de capacitación en recursos naturales
    ],
    "serie repetida (máx. 4 por serie)": [
        "20.500.12542/3427", "20.500.12542/4849",  # Paleoclima
    ],
    "fuera del dominio": [
        "20.500.12542/254",   # Beneficios socioeconómicos de los servicios climáticos
        "20.500.12542/2395",  # Caudales de la cuenca alta del Huallaga
    ],
    "informe operativo de obra, sin contenido glaciar": [
        "20.500.12543/2915",  # Presa de control de la laguna Conococha
    ],
    # POR REVISAR (no aplicado): con solo el título no se puede decidir. Si al ver el resumen o el
    # PDF no aportan a glaciares / alta montaña, quitar los # para excluirlos.
    # "revisado: fuera de alcance": [
    #     "20.500.12543/4255", "20.500.12543/4251", "20.500.12543/2876", "20.500.12543/1035",
    #     "20.500.12543/3166", "20.500.12543/2052", "20.500.12543/2975", "20.500.12543/442",
    #     "20.500.12543/46", "20.500.12543/43", "20.500.12543/42", "20.500.12543/41", "20.500.12543/40",
    #     "20.500.12543/15", "20.500.12543/5073", "20.500.12543/2031", "20.500.12543/4704",
    #     "20.500.12543/5714", "20.500.12543/211", "20.500.12543/4399", "20.500.12543/2081",
    #     "20.500.12543/4991", "20.500.12542/3085", "20.500.12542/3171", "20.500.12542/592",
    #     "20.500.12542/6199",
    # ],
}

# ── 3. Inclusiones forzadas (ganan sobre reglas y exclusiones) ──
INCLUIR = {
    "drenaje ácido de roca (consecuencia del retroceso glaciar)": [
        "20.500.12748/614", "20.500.12748/615", "20.500.12748/616",
    ],
    "ecosistemas de montaña": [
        "20.500.12748/631",
        "20.500.12748/475",   # líquenes en bosques de Polylepis
    ],
    "glaciar peruano (Shallap) comparado con la Antártida": [
        "20.500.12542/1908",
    ],
}

# ── Aplicar ──
decisiones, resumen_dec, no_estan = construir_decisiones(df_base, REGLAS_TITULO, EXCLUIR, INCLUIR, REVISOR)
RUTA_HISTORIAL = guardar_decisiones(decisiones, RUTA_DECISIONES)


def preparar(ruta_decisiones=RUTA_DECISIONES):
    # Correcciones → decisiones → selección balanceada
    d = aplicar_correcciones(df_base, CORRECCIONES, CONFIG["TIPOS_EXCLUIDOS"])
    d = aplicar_decisiones(d, cargar_decisiones(ruta_decisiones))
    d = seleccionar_corpus(d, CONFIG["TAMANO_CORPUS"], CONFIG["MIN_RELEVANCIA"],
                           CONFIG["PROPORCION_INVENTARIOS"], CONFIG["SOLO_CON_PDF"], CONFIG["MAX_POR_SERIE"])
    return d, d[d["seleccionado"]].copy()


df, corpus = preparar()

print(resumen_dec.to_string(index=False))
print(f"\nDecisiones guardadas: {len(decisiones)} filas · {decisiones.paper_id.nunique()} documentos")
if no_estan:
    print(f"⚠️ {len(no_estan)} handles no están entre los candidatos (¿fuente caída?):", no_estan[:10])

# Control: ¿el corpus es el mismo que con las decisiones de la v4 (celdas de parche)?
if RUTA_HISTORIAL:
    _, corpus_v4 = preparar(RUTA_HISTORIAL)
    solo_v4 = df[df.paper_id.isin(set(corpus_v4.paper_id) - set(corpus.paper_id))]
    solo_v5 = corpus[~corpus.paper_id.isin(corpus_v4.paper_id)]
    if solo_v4.empty and solo_v5.empty:
        print(f"\nControl v4 → v5: mismo corpus ({len(corpus)} documentos) ✅")
    else:
        print(f"\nControl v4 → v5: cambian {len(solo_v4)} documentos (revisar):")
        print("Solo en v4:"); display(solo_v4[["paper_id", "title", "fuente", "motivo_exclusion"]])
        print("Solo en v5:"); display(solo_v5[["paper_id", "title", "fuente"]])

In [ ]:
# Resumen del corpus resultante
pend = int(((corpus.exclusion == "revisar") & (corpus.decision_manual == "")).sum())
print(f"Corpus: {len(corpus)} documentos (objetivo {CONFIG['TAMANO_CORPUS']}) | pendientes de revisar: {pend}")
print("Por fuente:", corpus.fuente.value_counts().to_dict())
print("Por eje principal:", corpus.eje_principal.value_counts().to_dict())
print("Por año:", corpus.year.value_counts().sort_index().to_dict())
print("Documentos que mencionan cada eje:",
      {e: int(corpus.mecanismos.str.contains(e).sum()) for e in MECANISMOS_REGEX})
print(f"Inventarios: {int(corpus.es_inventario.sum())} | semillas: {int(corpus.semilla.sum())} | "
      f"con PDF descargable: {(corpus.pdf_estado == 'disponible').sum()}")

### Vista para revisar el corpus (opcional)
Muestra los seleccionados; los marcados `revisar` aparecen primero. Si alguno no corresponde, su `paper_id` va a `EXCLUIR` en la celda de decisiones.

In [ ]:
cols = ["paper_id", "title", "year", "eje_principal", "fuente", "exclusion", "decision_manual"]
vista = corpus.assign(_orden=(corpus.exclusion != "revisar")).sort_values(["_orden", "fuente", "year"])[cols]
with pd.option_context("display.max_rows", 600, "display.max_colwidth", 100):
    display(vista)

## 8. Guardar resultados y reporte de calidad
Archivos en la carpeta de trabajo: `papers.csv`, `candidatos.csv`, `authorships.csv`, `citations.csv`, `metadata_triples.csv`, `revision_pendiente.csv`, `README_datos.md`, `search_log.json`, `reporte_calidad.md`.

In [ ]:
def guardar():
    return guardar_resultados(CARPETA, corpus, df, registros, autorias, total_repo, resumen_fuentes,
                              CONFIG, SEMILLAS_HANDLES, codificador.metodo, cache, RUTA_DECISIONES)

salida = guardar()
for estado, msg in salida["checks"]:
    print(estado, msg)
print(f"\nTripletas por predicado: {salida['tripletas'].predicate.value_counts().to_dict()}")
print("Archivos guardados en", CARPETA)

## 9. Descargar PDFs
- Reanudable: los ya descargados no se repiten (solo bajan los nuevos de la ANA y el SENAMHI).
- Al terminar, **se vuelve a generar el reporte de la sección 8** para que incluya los PDFs.
- Los PDFs de ejecuciones anteriores que ya no están en el corpus se pueden apartar a `pdfs/_fuera_del_corpus` (no se borran) con `APARTAR_SOBRANTES = True`. Así la Persona 2 solo procesa los 398.

In [ ]:
DESCARGAR_PDFS = True
APARTAR_SOBRANTES = True
CARPETA_PDFS = os.path.join(CARPETA, "pdfs")
RUTA_MANIF = os.path.join(CARPETA, "pdfs_manifest.csv")

if DESCARGAR_PDFS:
    manif = descargar_pdfs(corpus, CARPETA_PDFS, RUTA_MANIF, CONFIG["REPO_BASE_URL"], cache)
    errores = manif[(manif.estado == "error") & manif.paper_id.isin(corpus.paper_id)]
    if len(errores):
        print(f"\n{len(errores)} PDFs con error (volver a ejecutar esta celda para reintentar):")
        display(errores[["paper_id", "error"]])

    sobran = pdfs_fuera_del_corpus(CARPETA_PDFS, corpus, mover=APARTAR_SOBRANTES)
    if sobran:
        print(f"\nPDFs que ya no están en el corpus: {len(sobran)}"
              + (" → movidos a pdfs/_fuera_del_corpus" if APARTAR_SOBRANTES else ""))

    salida = guardar()   # reporte actualizado con el estado de los PDFs
    print("\nReporte actualizado:")
    for estado, msg in salida["checks"]:
        print(estado, msg)

## 10. Resumen para la presentación

In [ ]:
c = salida["conteos"]
print("Documentos del tema por fuente:", {k: v["del_tema"] for k, v in resumen_fuentes.items()})
print(f"Candidatos evaluados: {len(df)} → excluidos {c['excluidos']} → corpus {len(corpus)}")
if corpus.year.notna().any():
    print(f"Periodo: {int(corpus.year.min())}–{int(corpus.year.max())}")
print("Corpus por fuente:", corpus.fuente.value_counts().to_dict())
print("Eje principal:", corpus.eje_principal.value_counts().to_dict())
print(f"Semillas: {int(corpus.semilla.sum())} | inventarios: {int(corpus.es_inventario.sum())}")
print(f"Autores únicos: {salida['autorias'].author_name.nunique()} | citas internas: {len(salida['citas'])} | "
      f"tripletas de metadatos: {len(salida['tripletas'])}")
print(f"Decisiones manuales: {decisiones.paper_id.nunique()} documentos | clasificador: {codificador.metodo}")
if os.path.exists(RUTA_MANIF):
    m = pd.read_csv(RUTA_MANIF, dtype={"paper_id": str})
    print(f"PDFs descargados del corpus: {((m.estado == 'ok') & m.paper_id.isin(corpus.paper_id)).sum()}/{len(corpus)}")

## Entrega
**Al repositorio (GitHub), carpeta `data/persona1/`:** este notebook, `papers.csv`, `authorships.csv`, `citations.csv`, `metadata_triples.csv`, `decisiones_manuales.csv`, `README_datos.md`, `search_log.json`, `reporte_calidad.md`, `pdfs_manifest.csv`.

**No se suben:** `pdfs/`, `cache/`, `candidatos.csv` (pesado; opcional).

**Para la Persona 2:** `papers.csv` (lista del corpus y metadatos) + la carpeta `pdfs/` en Drive. Cada PDF se llama como su handle con `_` (por ejemplo `20.500.12748_57.pdf`) y se cruza con `papers.csv` por `paper_id` usando `pdfs_manifest.csv`.

### Cambios de la v5
- Un solo notebook ordenado, sin celdas de parche.
- Todas las decisiones en **una sola celda** (reglas por título → excluir → incluir), y el CSV de decisiones se regenera desde ella.
- La primera ejecución respalda el CSV anterior y comprueba que el efecto sea el mismo.
- La sección 9 descarga los PDFs, aparta los que ya no están en el corpus y actualiza el reporte sola.